In [ ]:
# =============================================================================
# CELDA 1 - PROYECTO INTEGRADOR (Diplomado Energía Circular Inteligente 5.0 e IA)
# Carga (o genera) el dataset sintético "dataset_solar_popayan.csv" que consume
# la Celda 2. Si el archivo ya existe, lo reutiliza.
#
# Salida: /content/dataset_solar_popayan.csv  (o ./dataset_solar_popayan.csv)
#
# AVISO ÉTICO: dataset 100% SINTÉTICO, calibrado con parámetros públicos de
# Popayán (UPME-IDEAM, DANE, CREG). No contiene personas reales.
# Con datos reales aplicaría la Ley 1581 de 2012 y el Decreto 1377 de 2013.
# =============================================================================
import os, sys, glob, math, random, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

SEED = 42
np.random.seed(SEED)
random.seed(SEED)

IN_COLAB = "google.colab" in sys.modules
OUT_DIR = "/content" if IN_COLAB else "."
RUTA_CSV = os.path.join(OUT_DIR, "dataset_solar_popayan.csv")

# -----------------------------------------------------------------------------
# 1. Si ya existe el CSV, no se regenera
# -----------------------------------------------------------------------------
if os.path.exists(RUTA_CSV):
    df = pd.read_csv(RUTA_CSV, encoding="utf-8")
    print(f"✔ Dataset ya existía: {RUTA_CSV} -> {df.shape[0]} filas x {df.shape[1]} columnas")
else:
    # -------------------------------------------------------------------------
    # 2. En Colab: ofrecer subida opcional; si el usuario sube uno, se usa
    # -------------------------------------------------------------------------
    subido = None
    if IN_COLAB:
        try:
            from google.colab import files  # type: ignore
            print("Si tienes el CSV, súbelo ahora. Si cancelas, se generará un dataset sintético.")
            up = files.upload()
            if up:
                nombre = list(up.keys())[0]
                subido = os.path.join(OUT_DIR, "dataset_solar_popayan.csv")
                # Guardar con el nombre canónico que espera la Celda 2
                with open(subido, "wb") as fh:
                    fh.write(up[nombre])
                os.remove(nombre) if os.path.exists(nombre) and nombre != subido else None
                print(f"✔ CSV subido y guardado en {subido}")
        except Exception as e:
            print("  (subida no disponible:", str(e)[:80], "— se generará sintético)")

    if subido and os.path.exists(subido):
        df = pd.read_csv(subido, encoding="utf-8")
        print(f"✔ Dataset cargado desde archivo subido: {df.shape[0]} filas x {df.shape[1]} columnas")
    else:
        # ---------------------------------------------------------------------
        # 3. Generador sintético con TODAS las columnas que la Celda 2 espera
        # ---------------------------------------------------------------------
        print("⚙ Generando dataset sintético calibrado para Popayán...")
        N = 1000
        rg = np.random.default_rng(SEED)

        # --- Parámetros de referencia (Popayán, Cauca) -----------------------
        # Radiación: 4.0-4.5 kWh/m2/día; HSP 4.0-4.5; tarifa 700-950 COP/kWh
        RAD = (4.00, 4.50)
        HSP = (4.00, 4.50)
        TARIFA = (700, 950)
        COSTO_KWP = (3_800_000, 5_500_000)     # COP/kWp residencial 2026
        POT_PANEL = [400, 430, 450, 500, 525, 550]
        AREA_PANEL_M2 = 1.95                    # ~1.7 x 1.1 m (panel 450 W típico)

        # --- Atributos socioeconómicos ---------------------------------------
        estrato = rg.choice([1, 2, 3], size=N, p=[0.65, 0.20, 0.15])
        zona = np.where(rg.random(N) < 0.78, "Urbana", "Rural")
        tipo_vivienda = np.where(
            zona == "Rural", "Vivienda rural",
            np.where(rg.random(N) < 0.75, "Casa", "Apartamento")
        )
        comuna = np.where(zona == "Urbana",
                          rg.integers(1, 10, size=N).astype(str),
                          "Corregimiento")
        barrio = np.where(zona == "Urbana",
                          [f"Barrio_{i:02d}" for i in rg.integers(1, 40, size=N)],
                          [f"Vereda_{i:02d}" for i in rg.integers(1, 30, size=N)])

        # --- Composición del hogar -------------------------------------------
        num_personas = rg.integers(1, 8, size=N)
        num_menores = np.array([rg.integers(0, min(4, p) + 1) for p in num_personas])
        num_adultos_mayores = np.array([
            rg.integers(0, min(2, p - m)) if (p - m) > 1 else 0
            for p, m in zip(num_personas, num_menores)
        ])

        # --- Ingreso según estrato (COP/mes) ---------------------------------
        ingreso_base = {1: 1_100_000, 2: 2_000_000, 3: 3_500_000}
        ingreso = np.array([
            max(500_000, rg.normal(ingreso_base[e], ingreso_base[e] * 0.35))
            for e in estrato
        ]).round(-3)

        jefe_genero = rg.choice(["Femenino", "Masculino"], size=N, p=[0.48, 0.52])
        nivel_educativo = rg.choice(
            ["Primaria", "Secundaria", "Técnico", "Universitario", "Posgrado"],
            size=N, p=[0.25, 0.35, 0.18, 0.17, 0.05]
        )
        ocupacion = rg.choice(
            ["Empleado", "Independiente", "Desempleado", "Pensionado", "Estudiante", "Hogar"],
            size=N, p=[0.35, 0.30, 0.08, 0.10, 0.07, 0.10]
        )
        tiene_empresa = rg.choice(["Sí", "No"], size=N, p=[0.15, 0.85])

        # --- Vivienda y cubierta ---------------------------------------------
        area_vivienda = np.array([
            max(30, rg.normal(70 if z == "Urbana" else 110, 25)) for z in zona
        ]).round(1)
        area_cubierta = (area_vivienda * rg.uniform(0.75, 1.15, size=N)).round(1)
        tipo_cubierta = rg.choice(
            ["Teja de barro", "Teja metálica", "Teja de zinc", "Losa de concreto", "Eternit"],
            size=N, p=[0.30, 0.25, 0.15, 0.20, 0.10]
        )
        inclinacion = np.where(tipo_cubierta == "Losa de concreto",
                               rg.uniform(2, 10, size=N),
                               rg.uniform(10, 30, size=N)).round(1)
        orientacion = rg.choice(["Norte", "Sur", "Oriente", "Occidente"],
                                size=N, p=[0.35, 0.25, 0.20, 0.20])
        sombras = rg.choice(["No", "Sí"], size=N, p=[0.72, 0.28])
        sombras_desc = np.where(
            sombras == "Sí",
            rg.choice(["Árboles", "Edificios vecinos", "Antenas", "Cables"],
                      size=N, p=[0.45, 0.30, 0.15, 0.10]),
            "Ninguna"
        )

        # --- Consumo eléctrico -----------------------------------------------
        # Base por persona, ajustado por estrato, zona, clima y equipos
        base_pp = {1: 35, 2: 55, 3: 85}       # kWh/mes por persona
        consumo = np.array([
            max(40, rg.normal(base_pp[e] * p, base_pp[e] * p * 0.30))
            for e, p in zip(estrato, num_personas)
        ])
        consumo *= np.where(zona == "Rural", 0.85, 1.0)       # menor consumo rural
        consumo = consumo.round(2)
        consumo_dia = (consumo / 30).round(3)

        # --- Perfil horario (24 h) normalizado -------------------------------
        # Curva típica residencial: pico mañana (7-8h) y pico noche (19-21h)
        horas = np.array([f"h{h:02d}" for h in range(24)])
        perfil_base = np.array([
            0.018, 0.015, 0.014, 0.014, 0.015, 0.020,   # 0-5  madrugada
            0.035, 0.055, 0.060, 0.045, 0.038, 0.040,   # 6-11 mañana
            0.045, 0.040, 0.038, 0.038, 0.042, 0.048,   # 12-17 tarde
            0.060, 0.070, 0.065, 0.055, 0.040, 0.025    # 18-23 noche
        ])
        perfil_base = perfil_base / perfil_base.sum()        # suma 1
        perfil_hogar = np.array([
            rg.dirichlet(perfil_base * 200) for _ in range(N)
        ])  # (N, 24)
        kWh_horas = (perfil_hogar * consumo_dia[:, None]).round(4)
        df_horas = pd.DataFrame(kWh_horas, columns=horas)

        # --- Demanda máxima y factor de carga --------------------------------
        factor_carga = rg.uniform(0.35, 0.65, size=N)
        demanda_max = (consumo_dia * 24 * factor_carga).round(3)

        # --- Factura y tarifa ------------------------------------------------
        tarifa = np.array([rg.uniform(*TARIFA) for _ in range(N)]).round(0)
        costo_factura = (consumo * tarifa).round(0)

        # --- Radiación, HSP y dimensionamiento FV ----------------------------
        radiacion = np.array([rg.uniform(*RAD) for _ in range(N)]).round(2)
        hsp = np.array([rg.uniform(*HSP) for _ in range(N)]).round(2)
        # Cobertura objetivo: 60-100% del consumo
        cobertura_obj = rg.uniform(0.60, 1.00, size=N)
        kwh_necesarios_dia = consumo_dia * cobertura_obj
        kwp = np.clip(kwh_necesarios_dia / (hsp * 0.78), 0.5, 6.0)
        pot_panel = rg.choice(POT_PANEL, size=N)
        n_paneles = np.maximum(1, np.ceil((kwp * 1000) / pot_panel).astype(int))
        kwp_real = (n_paneles * pot_panel / 1000).round(2)
        area_req = (n_paneles * AREA_PANEL_M2).round(1)

        gen_dia = (kwp_real * hsp * 0.78).round(2)
        gen_mes = (gen_dia * 30).round(1)

        # --- Económicos ------------------------------------------------------
        costo_kwp = np.array([rg.uniform(*COSTO_KWP) for _ in range(N)]).round(-3)
        inversion = (kwp_real * costo_kwp).round(-3)
        ahorro_mes = np.minimum(costo_factura * cobertura_obj, gen_mes * tarifa).round(0)
        ahorro_anual = (ahorro_mes * 12).round(0)
        pct_ahorro = (100 * ahorro_mes / np.maximum(costo_factura, 1)).round(1)
        retorno_meses = (inversion / np.maximum(ahorro_mes, 1)).round(1)
        retorno_anios = (retorno_meses / 12).round(2)
        # VPN 10 años al 12% anual (tarifa creciente 5%/año, degradación 0.5%/año)
        tasa = 0.12
        flujos = [ahorro_anual * ((1.05 * 0.995) ** t) for t in range(10)]
        vpn = float(np.mean([sum(f / (1 + tasa) ** (t + 1) for t, f in enumerate(flujos)) - inversion[i]
                              for i in range(3)])) if False else 0.0
        vpn_arr = np.zeros(N)
        for i in range(N):
            fl = [ahorro_anual[i] * ((1.05 * 0.995) ** t) for t in range(10)]
            vpn_arr[i] = sum(f / (1 + tasa) ** (t + 1) for t, f in enumerate(fl)) - inversion[i]
        vpn = vpn_arr.round(0)
        tir_arr = np.zeros(N)
        for i in range(N):
            fl = [-inversion[i]] + [ahorro_anual[i] * ((1.05 * 0.995) ** t) for t in range(10)]
            try:
                tir_arr[i] = (np.irr(fl) if hasattr(np, "irr") else np.roots(fl).real[0]) * 100
            except Exception:
                tir_arr[i] = 0.0
        # np.irr fue removido en NumPy 1.20+; usamos bisección rápida
        def _irr(flujos, lo=-0.9, hi=2.0, tol=1e-4):
            def npv(r):
                return sum(f / (1 + r) ** t for t, f in enumerate(flujos))
            if npv(lo) * npv(hi) > 0:
                return 0.0
            for _ in range(80):
                mid = (lo + hi) / 2
                if npv(lo) * npv(mid) <= 0:
                    hi = mid
                else:
                    lo = mid
                if abs(hi - lo) < tol:
                    break
            return (lo + hi) / 2
        tir_arr = np.array([_irr([-inversion[i]] + [ahorro_anual[i] * ((1.05 * 0.995) ** t) for t in range(10)]) * 100
                             for i in range(N)]).round(2)
        tir_arr = np.clip(tir_arr, -50, 200)

        requiere_fin = np.where(inversion > 4 * ingreso, "Sí", "No")
        fuente_fin = np.where(requiere_fin == "Sí",
                              rg.choice(["Bancoldex", "Crédito de consumo", "Fondo municipal", "Cooperativa"],
                                        size=N, p=[0.30, 0.35, 0.20, 0.15]),
                              "No aplica")

        # --- Tecnología y apropiación digital --------------------------------
        tiene_internet = rg.choice(["Sí", "No"], size=N, p=[0.62, 0.38])
        tiene_smart = rg.choice(["Sí", "No"], size=N, p=[0.78, 0.22])
        nivel_tecno = rg.choice(["Bajo", "Medio", "Alto"], size=N, p=[0.35, 0.45, 0.20])
        dispositivo_ia = np.where(
            (tiene_internet == "No") & (tiene_smart == "No"),
            "Ninguno",
            rg.choice(["Asistente de voz", "App móvil", "Chatbot", "Ninguno"],
                      size=N, p=[0.35, 0.40, 0.15, 0.10])
        )
        medidor_int = rg.choice(["Sí", "No"], size=N, p=[0.18, 0.82])
        recibe_sub = np.where(estrato == 1, "Sí",
                              np.where(estrato == 2, rg.choice(["Sí", "No"], size=N, p=[0.55, 0.45]), "No"))
        reciclaje_conoce = rg.choice(["Sí", "No"], size=N, p=[0.30, 0.70])
        comunidad_int = rg.choice(["Sí", "No"], size=N, p=[0.42, 0.58])

        # --- Decisión de adopción (target) ----------------------------------
        # Función latente: depende de ahorro, retorno, ingreso, educación, internet...
        nivel_edu_num = pd.Series(nivel_educativo).map(
            {"Primaria": 0, "Secundaria": 1, "Técnico": 2, "Universitario": 3, "Posgrado": 4}
        ).values
        logit = (
            0.8 * (pct_ahorro / 100)
            - 0.15 * retorno_anios
            + 0.35 * nivel_edu_num
            + 0.40 * (tiene_internet == "Sí").astype(float)
            + 0.30 * (tiene_smart == "Sí").astype(float)
            + 0.45 * (nivel_tecno == "Alto").astype(float)
            + 0.25 * (comunidad_int == "Sí").astype(float)
            - 0.35 * (estrato == 1).astype(float)
            + 0.20 * (vpn > 0).astype(float)
            + rg.normal(0, 0.9, N)
        )
        p_si = 1 / (1 + np.exp(-logit))
        u = rg.random(N)
        dispuesto = np.where(u < p_si * 0.55, "Sí",
                              np.where(u < p_si * 0.55 + 0.30, "Tal vez", "No"))

        # --- Actitudes post-decisión (NO se usan como predictores) -----------
        barreras = rg.choice(
            ["Costo inicial alto", "Falta de información", "Desconfianza en la tecnología",
             "Falta de espacio", "Trámites complejos", "Falta de financiación"],
            size=N, p=[0.34, 0.18, 0.14, 0.10, 0.10, 0.14]
        )
        motivaciones = rg.choice(
            ["Ahorro económico", "Cuidado del ambiente", "Independencia energética",
             "Modernización del hogar", "Presión de la comunidad"],
            size=N, p=[0.50, 0.22, 0.12, 0.10, 0.06]
        )
        disp_pago = np.where(dispuesto == "No", 0,
                             np.maximum(0, rg.normal(ahorro_mes * 0.4, ahorro_mes * 0.15)).round(-3))
        prioridad_gob = rg.choice(
            ["Subsidios", "Financiación blanda", "Capacitación", "Instalación gratuita", "Ninguna"],
            size=N, p=[0.30, 0.25, 0.15, 0.20, 0.10]
        )

        # --- Variación estacional y meses de mayor/menor consumo -------------
        variacion_est = rg.uniform(5, 25, size=N).round(1)
        meses = ["Enero", "Febrero", "Marzo", "Abril", "Mayo", "Junio",
                 "Julio", "Agosto", "Septiembre", "Octubre", "Noviembre", "Diciembre"]
        mes_mayor = rg.choice(["Abril", "Mayo", "Octubre", "Noviembre", "Diciembre"], size=N)
        mes_menor = rg.choice(["Enero", "Febrero", "Julio", "Agosto"], size=N)

        # --- Equipos electrodomésticos ---------------------------------------
        num_neveras = rg.integers(0, 3, size=N)
        pot_nevera = np.where(num_neveras > 0, rg.choice([150, 200, 250], size=N), 0)
        num_tv = rg.integers(0, 4, size=N)
        pot_tv = np.where(num_tv > 0, rg.choice([60, 80, 100, 120], size=N), 0)
        num_lav = rg.integers(0, 2, size=N)
        pot_lav = np.where(num_lav > 0, rg.choice([400, 500, 600], size=N), 0)
        num_vent = rg.integers(0, 4, size=N)
        pot_vent = np.where(num_vent > 0, rg.choice([50, 60, 75], size=N), 0)
        num_bomb = rg.integers(2, 16, size=N)
        pot_bomb = rg.choice([9, 12, 15], size=N)
        num_comp = rg.integers(0, 3, size=N)
        num_esp = rg.integers(0, 2, size=N)

        # --- Ensamblado del DataFrame ----------------------------------------
        df = pd.DataFrame({
            "id_hogar": [f"H{i:04d}" for i in range(1, N + 1)],
            "comuna": comuna,
            "barrio": barrio,
            "estrato": estrato,
            "tipo_vivienda": tipo_vivienda,
            "zona": zona,
            "num_personas": num_personas,
            "num_menores": num_menores,
            "num_adultos_mayores": num_adultos_mayores,
            "ingreso_mensual_cop": ingreso,
            "jefe_hogar_genero": jefe_genero,
            "nivel_educativo": nivel_educativo,
            "ocupacion_principal": ocupacion,
            "tiene_empresa": tiene_empresa,
            "area_vivienda_m2": area_vivienda,
            "area_cubierta_m2": area_cubierta,
            "tipo_cubierta": tipo_cubierta,
            "inclinacion_cubierta_grados": inclinacion,
            "orientacion_cubierta": orientacion,
            "sombras_presentes": sombras,
            "sombras_descripcion": sombras_desc,
            "consumo_kwh_mes": consumo,
            "consumo_kwh_dia_promedio": consumo_dia,
            "demanda_maxima_kw": demanda_max,
            "factor_carga": factor_carga.round(3),
            "costo_factura_mes_cop": costo_factura,
            "tarifa_kwh_cop": tarifa,
            "consumo_anual_kwh": (consumo * 12).round(1),
            "variacion_estacional_pct": variacion_est,
            "mes_mayor_consumo": mes_mayor,
            "mes_menor_consumo": mes_menor,
            "tiene_medidor_inteligente": medidor_int,
            "recibe_subsidio": recibe_sub,
            "radiacion_promedio_kwh_m2_dia": radiacion,
            "horas_sol_pico": hsp,
            "generacion_potencial_kwp": kwp_real,
            "num_paneles_estimados": n_paneles,
            "potencia_panel_w": pot_panel,
            "area_requerida_m2": area_req,
            "generacion_diaria_estimada_kwh": gen_dia,
            "generacion_mensual_estimada_kwh": gen_mes,
            "inversion_estimada_cop": inversion,
            "costo_kwp_cop": costo_kwp,
            "ahorro_mensual_cop": ahorro_mes,
            "ahorro_anual_cop": ahorro_anual,
            "porcentaje_ahorro": pct_ahorro,
            "retorno_simple_meses": retorno_meses,
            "retorno_simple_anios": retorno_anios,
            "vpn_10_anios_cop": vpn,
            "tir_porcentaje": tir_arr,
            "requiere_financiacion": requiere_fin,
            "fuente_financiacion": fuente_fin,
            "dispositivo_ia_interes": dispositivo_ia,
            "nivel_apropiacion_tecnologica": nivel_tecno,
            "tiene_internet": tiene_internet,
            "tiene_smartphone": tiene_smart,
            "interes_comunidad_energetica": comunidad_int,
            "conoce_reciclaje_paneles": reciclaje_conoce,
            "dispuesto_instalar_paneles": dispuesto,
            "principal_barrera": barreras,
            "principal_motivacion": motivaciones,
            "disposicion_pago_mensual_cop": disp_pago,
            "prioridad_gobierno": prioridad_gob,
            "num_neveras": num_neveras,
            "potencia_nevera_w": pot_nevera,
            "num_televisores": num_tv,
            "potencia_tv_w": pot_tv,
            "num_lavadoras": num_lav,
            "potencia_lavadora_w": pot_lav,
            "num_ventiladores": num_vent,
            "potencia_ventilador_w": pot_vent,
            "num_bombillas_led": num_bomb,
            "potencia_bombilla_w": pot_bomb,
            "num_computadores": num_comp,
            "num_equipos_especiales": num_esp,
        })

        # Unir las 24 columnas de consumo horario
        df = pd.concat([df, df_horas], axis=1)

        # Pequeño ajuste físico: recortar facturas inviables
        df["ahorro_mensual_cop"] = np.minimum(df["ahorro_mensual_cop"], df["costo_factura_mes_cop"])
        df["ahorro_anual_cop"] = (df["ahorro_mensual_cop"] * 12).round(0)
        df["porcentaje_ahorro"] = (100 * df["ahorro_mensual_cop"] / df["costo_factura_mes_cop"]).round(1)
        df["retorno_simple_meses"] = (df["inversion_estimada_cop"] / np.maximum(df["ahorro_mensual_cop"], 1)).round(1)
        df["retorno_simple_anios"] = (df["retorno_simple_meses"] / 12).round(2)
        # VPN coherente con TIR: si VPN>0 => TIR>10%
        df.loc[df["vpn_10_anios_cop"] <= 0, "tir_porcentaje"] = np.minimum(
            df.loc[df["vpn_10_anios_cop"] <= 0, "tir_porcentaje"], 10.0
        )
        df.loc[df["vpn_10_anios_cop"] > 0, "tir_porcentaje"] = np.maximum(
            df.loc[df["vpn_10_anios_cop"] > 0, "tir_porcentaje"], 10.01
        )

        # Guardar
        os.makedirs(OUT_DIR, exist_ok=True)
        df.to_csv(RUTA_CSV, index=False, encoding="utf-8")
        print(f"✔ Dataset sintético generado y guardado en: {RUTA_CSV}")
        print(f"  Filas: {df.shape[0]} | Columnas: {df.shape[1]}")
        print(f"  Balance del target 'dispuesto_instalar_paneles':")
        print("  ", df['dispuesto_instalar_paneles'].value_counts(normalize=True).round(3).to_dict())

# -----------------------------------------------------------------------------
# 4. Verificación mínima de columnas que la Celda 2 espera
# -----------------------------------------------------------------------------
HORAS = [f"h{h:02d}" for h in range(24)]
COLS_CRITICAS = [
    "id_hogar", "estrato", "zona", "consumo_kwh_mes", "tarifa_kwh_cop",
    "costo_factura_mes_cop", "ahorro_anual_cop", "dispuesto_instalar_paneles",
    "dispositivo_ia_interes", "jefe_hogar_genero",
] + HORAS
faltantes = [c for c in COLS_CRITICAS if c not in df.columns]
if faltantes:
    print(f"⚠ Faltan columnas críticas en el CSV: {faltantes}")
else:
    print("✔ Verificación OK: todas las columnas críticas para la Celda 2 están presentes.")

print("\nPrimeras filas:")
print(df.head(3).T.to_string())

Si tienes el CSV, súbelo ahora. Si cancelas, se generará un dataset sintético.


Saving dataset_solar_popayan (1).csv to dataset_solar_popayan (1).csv
✔ CSV subido y guardado en /content/dataset_solar_popayan.csv
✔ Dataset cargado desde archivo subido: 1000 filas x 99 columnas
✔ Verificación OK: todas las columnas críticas para la Celda 2 están presentes.

Primeras filas:
                                                    0                 1                2
id_hogar                                        H0001             H0002            H0003
comuna                                      Yanaconas           Calibío         Comuna 2
barrio                               Vereda Primavera  Vereda Primavera     Los Naranjos
estrato                                             2                 1                1
tipo_vivienda                          Vivienda rural    Vivienda rural             Casa
zona                                            Rural             Rural           Urbana
num_personas                                        1                 6            

In [ ]:
# =============================================================================
# CELDA 2 - PROYECTO INTEGRADOR (Diplomado Energía Circular Inteligente 5.0 e IA)
# "Autogeneración Solar Fotovoltaica en Hogares de Popayán, Cauca (2026-2030)"
# Pipeline: ETL -> EDA -> ML supervisado (clásicos + RED NEURONAL MULTIMODAL PyTorch)
#           -> ML no supervisado -> series temporales -> optimización
#           -> economía circular -> dashboard Gradio -> exportación
# =============================================================================
import os, sys, io, re, glob, json, math, time, shutil, zipfile, subprocess, warnings, itertools, pickle
warnings.filterwarnings("ignore")
T0 = time.time()

def _asegurar_paquetes():
    import importlib
    requeridos = {"gradio": "gradio", "plotly": "plotly", "shap": "shap", "statsmodels": "statsmodels",
                  "prophet": "prophet", "sklearn": "scikit-learn",
                  "lightgbm": "lightgbm", "pulp": "pulp", "kaleido": "kaleido", "scipy": "scipy", "joblib": "joblib"}
    faltan = []
    for mod, pipname in requeridos.items():
        try:
            m_ = importlib.import_module(mod)
            if mod == "pulp" and int(str(m_.__version__).split(".")[0]) >= 4:
                faltan.append("pulp<4")
        except Exception:
            faltan.append("pulp<4" if mod == "pulp" else pipname)
    try:
        import torch  # noqa
    except Exception:
        faltan.append("torch")
    if faltan:
        print("Instalando paquetes faltantes:", faltan)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + faltan, check=False)
_asegurar_paquetes()

import numpy as np
import pandas as pd
import joblib
import plotly.express as px
import plotly.graph_objects as go
import plotly.figure_factory as ff
import plotly.io as pio
from plotly.subplots import make_subplots
from scipy import stats
from scipy.cluster.hierarchy import linkage
import gradio as gr

from sklearn.base import clone, BaseEstimator, ClassifierMixin
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder, FunctionTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import (train_test_split, StratifiedKFold, KFold, cross_validate,
                                     cross_val_predict, RandomizedSearchCV)
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, StackingClassifier, IsolationForest
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier, MLPRegressor
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.inspection import permutation_importance
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score, roc_auc_score, roc_curve,
                             average_precision_score, precision_recall_curve, matthews_corrcoef, cohen_kappa_score,
                             log_loss, confusion_matrix, r2_score, mean_absolute_error, mean_squared_error,
                             silhouette_score, calinski_harabasz_score, davies_bouldin_score, adjusted_rand_score,
                             precision_score, recall_score)
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.neighbors import NearestNeighbors
from lightgbm import LGBMClassifier
import shap
import pulp
import statsmodels.api as sm
from statsmodels.tsa.seasonal import STL
from statsmodels.tsa.stattools import acf, pacf
from statsmodels.tsa.statespace.sarimax import SARIMAX
try:
    from prophet import Prophet
    PROPHET_OK = True
except Exception as _e:
    PROPHET_OK = False
    print("AVISO: Prophet no disponible (", _e, "). Se continuará con SARIMAX y línea base estacional.")

try:
    import torch
    import torch.nn as nn
    NN_OK = True
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"[NN] PyTorch {torch.__version__} disponible. Device = {DEVICE}")
except Exception as _e:
    NN_OK = False
    DEVICE = None
    print("[NN] PyTorch NO disponible (", str(_e)[:80], "). Se usará MLP de sklearn como fallback.")

import logging
for _lg in ("cmdstanpy", "prophet", "prophet.plot", "matplotlib", "numexpr", "lightgbm"):
    logging.getLogger(_lg).setLevel(logging.ERROR)
SEED = 42
np.random.seed(SEED)
pio.templates.default = "plotly_white"
LAUNCH = os.environ.get("NO_LAUNCH", "0") != "1"

if NN_OK:
    torch.manual_seed(SEED)
    try:
        torch.use_deterministic_algorithms(True, warn_only=True)
    except Exception:
        pass

IN_COLAB = "google.colab" in sys.modules
OUT = "/content/outputs" if IN_COLAB else "outputs"
DIRS = {"base": OUT, "graficos": f"{OUT}/graficos", "tablas": f"{OUT}/tablas", "modelos": f"{OUT}/modelos"}
for _d in DIRS.values():
    os.makedirs(_d, exist_ok=True)

print("=" * 100)
print("AVISO ÉTICO: dataset SINTÉTICO. Con datos reales aplicaría Ley 1581 de 2012 y Decreto 1377 de 2013.")
print("=" * 100)

FIGS, TABLAS, MODELOS = {}, {}, {}
_PNG_OK = {"estado": None}

def reg_fig(nombre, fig, clave=False):
    FIGS[nombre] = fig
    try:
        fig.write_html(f"{DIRS['graficos']}/{nombre}.html", include_plotlyjs="cdn")
    except Exception as e:
        print("  (no se pudo exportar HTML de", nombre, ":", e, ")")
    if clave and _PNG_OK["estado"] is not False:
        try:
            fig.write_image(f"{DIRS['graficos']}/{nombre}.png", width=1100, height=650, scale=1.5)
            _PNG_OK["estado"] = True
        except Exception:
            ok = False
            if _PNG_OK["estado"] is None:
                try:
                    pio.get_chrome()
                    fig.write_image(f"{DIRS['graficos']}/{nombre}.png", width=1100, height=650, scale=1.5)
                    ok = True
                except Exception:
                    ok = False
            _PNG_OK["estado"] = ok
            if not ok:
                print("  (PNG no disponible; se exportan los gráficos en HTML interactivo)")
    return fig

def reg_tabla(nombre, df_, index=False):
    TABLAS[nombre] = df_
    df_.to_csv(f"{DIRS['tablas']}/{nombre}.csv", index=index, encoding="utf-8-sig")
    return df_

def log(msg):
    print(f"[{time.time() - T0:7.1f}s] {msg}")

# =============================================================================
# BLOQUE 1. ETL Y FEATURE ENGINEERING
# =============================================================================
log("BLOQUE 1: ETL y Feature Engineering")

def localizar_csv():
    nombres = ["dataset_solar_popayan*.csv"]
    carpetas = [".", "/content", "/content/sample_data", "/mnt/user-data/outputs", "/mnt/user-data/uploads"]
    for c in carpetas:
        for n in nombres:
            r = sorted(glob.glob(os.path.join(c, n)))
            if r:
                return r[-1]
    raise FileNotFoundError("No se encontró dataset_solar_popayan.csv. Ejecuta primero la celda 1.")

RUTA_CSV = localizar_csv()
df_raw = pd.read_csv(RUTA_CSV, encoding="utf-8")
print(f"Dataset cargado: {RUTA_CSV} -> {df_raw.shape[0]} filas x {df_raw.shape[1]} columnas")

HORAS = [f"h{h:02d}" for h in range(24)]
COLS_ESPERADAS = ["id_hogar", "estrato", "zona", "consumo_kwh_mes", "dispuesto_instalar_paneles", "ahorro_anual_cop"] + HORAS
_faltan = [c for c in COLS_ESPERADAS if c not in df_raw.columns]
if _faltan:
    raise ValueError(f"Faltan columnas esperadas en el CSV: {_faltan}")

def tabla_calidad(d):
    filas = []
    def chk(regla, ok_mask):
        filas.append([regla, int((~ok_mask).sum()), f"{100 * ok_mask.mean():.1f}%"])
    chk("Sin valores nulos (por fila)", ~d.isna().any(axis=1))
    chk("id_hogar único", ~d.duplicated("id_hogar", keep=False))
    chk("Sin filas duplicadas", ~d.duplicated(keep=False))
    chk("Suma h00-h23 = consumo diario", np.isclose(d[HORAS].sum(axis=1), d["consumo_kwh_dia_promedio"], atol=1e-3))
    chk("Factura = consumo x tarifa", (d["consumo_kwh_mes"] * d["tarifa_kwh_cop"] - d["costo_factura_mes_cop"]).abs() <= 1)
    chk("Ahorro mensual <= factura", d["ahorro_mensual_cop"] <= d["costo_factura_mes_cop"])
    chk("Área requerida <= área cubierta", d["area_requerida_m2"] <= d["area_cubierta_m2"])
    chk("Consumo anual = 12 x mensual", (d["consumo_anual_kwh"] - 12 * d["consumo_kwh_mes"]).abs() < 0.2)
    chk("Menores + mayores < personas", (d["num_menores"] + d["num_adultos_mayores"]) < d["num_personas"])
    chk("Estrato 3 sin subsidio", ~((d["estrato"] == 3) & (d["recibe_subsidio"] == "Sí")))
    chk("Sin internet ni smartphone => IA 'Ninguno'",
        ~((d["tiene_internet"] == "No") & (d["tiene_smartphone"] == "No") & (d["dispositivo_ia_interes"] != "Ninguno")))
    chk("VPN>0 coherente con TIR>10%", (d["vpn_10_anios_cop"] > 0) == (d["tir_porcentaje"] > 10))
    return pd.DataFrame(filas, columns=["regla_de_coherencia", "filas_que_incumplen", "cumplimiento"])

calidad = reg_tabla("calidad_datos", tabla_calidad(df_raw))
print(calidad.to_string(index=False))

COLS_WINSOR = ["ingreso_mensual_cop", "demanda_maxima_kw", "inversion_estimada_cop", "retorno_simple_meses",
               "retorno_simple_anios", "vpn_10_anios_cop", "tir_porcentaje"]
df = df_raw.copy()
resumen_w = []
for c in COLS_WINSOR:
    lo, hi = df[c].quantile(0.01), df[c].quantile(0.99)
    n_mod = int(((df[c] < lo) | (df[c] > hi)).sum())
    df[c] = df[c].clip(lo, hi)
    resumen_w.append([c, round(lo, 3), round(hi, 3), n_mod])
reg_tabla("winsorizacion", pd.DataFrame(resumen_w, columns=["variable", "p01", "p99", "valores_ajustados"]))

HSP_REF, PR_REF = 4.25, 0.78
_tmp = df.assign(_pp=df["consumo_kwh_mes"] / df["num_personas"], _m2=df["consumo_kwh_mes"] / df["area_vivienda_m2"])
REF_PP = _tmp.groupby("estrato")["_pp"].median().to_dict()
REF_M2 = _tmp.groupby("estrato")["_m2"].median().to_dict()

def ingenieria_caracteristicas(d):
    d = d.copy()
    d["kwp_calc"] = d["num_paneles_estimados"] * d["potencia_panel_w"] / 1000.0
    d["carga_familiar"] = (d["num_menores"] + d["num_adultos_mayores"]) / d["num_personas"]
    d["ratio_consumo_ingreso_pct"] = 100.0 * d["consumo_kwh_mes"] * d["tarifa_kwh_cop"] / d["ingreso_mensual_cop"]
    d["kwh_por_persona"] = d["consumo_kwh_mes"] / d["num_personas"]
    d["kwh_por_m2"] = d["consumo_kwh_mes"] / d["area_vivienda_m2"]
    rpp = d["estrato"].map(REF_PP); rm2 = d["estrato"].map(REF_M2)
    d["indice_eficiencia_vivienda"] = (100.0 * (1 - 0.5 * (d["kwh_por_persona"] / (2 * rpp) + d["kwh_por_m2"] / (2 * rm2)))).clip(0, 100)
    d["cobertura_solar_pct"] = 100.0 * d["kwp_calc"] * HSP_REF * 30 * PR_REF / d["consumo_kwh_mes"]
    d["brecha_digital"] = (d["tiene_internet"] == "No").astype(int) + (d["tiene_smartphone"] == "No").astype(int)
    return d

def agregar_perfil(d):
    d = d.copy()
    tot = d[HORAS].sum(axis=1)
    d["ratio_diurno_6_17"] = d[[f"h{h:02d}" for h in range(6, 18)]].sum(axis=1) / tot
    d["ratio_manana_6_8"] = d[["h06", "h07", "h08"]].sum(axis=1) / tot
    d["ratio_mediodia_12_14"] = d[["h12", "h13", "h14"]].sum(axis=1) / tot
    d["ratio_noche_18_22"] = d[[f"h{h:02d}" for h in range(18, 23)]].sum(axis=1) / tot
    d["ratio_madrugada_0_5"] = d[[f"h{h:02d}" for h in range(0, 6)]].sum(axis=1) / tot
    d["factor_pico_horario"] = d[HORAS].max(axis=1) / d[HORAS].mean(axis=1)
    return d

df = agregar_perfil(ingenieria_caracteristicas(df))
df["objetivo_adopcion"] = df["dispuesto_instalar_paneles"]
CLASES = ["No", "Tal vez", "Sí"]
print("Características creadas correctamente.")

DESC = {
 "id_hogar": "Identificador único del hogar", "estrato": "Estrato socioeconómico (1-3)",
 "zona": "Urbana o Rural", "num_personas": "Personas en el hogar",
 "num_menores": "Menores de edad", "num_adultos_mayores": "Adultos mayores",
 "ingreso_mensual_cop": "Ingreso mensual del hogar (COP)",
 "area_cubierta_m2": "Área de cubierta disponible (m²)", "tipo_cubierta": "Material de la cubierta",
 "sombras_presentes": "Sombras sobre la cubierta", "consumo_kwh_mes": "Consumo eléctrico mensual (kWh)",
 "consumo_kwh_dia_promedio": "Consumo diario promedio (kWh)", "tarifa_kwh_cop": "Tarifa efectiva (COP/kWh)",
 "generacion_potencial_kwp": "Potencia FV dimensionada (kWp)", "num_paneles_estimados": "Número de paneles",
 "potencia_panel_w": "Potencia por panel (W)", "inversion_estimada_cop": "Inversión estimada (COP)",
 "ahorro_anual_cop": "Ahorro anual (COP)", "porcentaje_ahorro": "Ahorro como % de la factura",
 "retorno_simple_anios": "Retorno simple (años)", "vpn_10_anios_cop": "VPN a 10 años (COP)",
 "tir_porcentaje": "TIR a 10 años (%)", "tiene_internet": "Tiene internet",
 "tiene_smartphone": "Tiene smartphone", "dispuesto_instalar_paneles": "OBJETIVO: Sí / Tal vez / No",
 "principal_barrera": "Principal barrera de adopción",
 "carga_familiar": "[FE]", "ratio_consumo_ingreso_pct": "[FE]", "kwh_por_persona": "[FE]", "kwh_por_m2": "[FE]",
 "indice_eficiencia_vivienda": "[FE]", "cobertura_solar_pct": "[FE]", "brecha_digital": "[FE]", "kwp_calc": "[FE]",
 "ratio_diurno_6_17": "[FE]", "ratio_manana_6_8": "[FE]", "ratio_mediodia_12_14": "[FE]", "ratio_noche_18_22": "[FE]",
 "ratio_madrugada_0_5": "[FE]", "factor_pico_horario": "[FE]", "objetivo_adopcion": "Copia del objetivo"}
for h in HORAS:
    DESC[h] = f"Consumo medio en la hora {h[1:]}:00 (kWh)"

def construir_diccionario(d):
    filas = []
    for c in d.columns:
        s = d[c]
        if pd.api.types.is_numeric_dtype(s):
            tipo = "numérica"; rango = f"{s.min():.4g} a {s.max():.4g}"
        else:
            tipo = "categórica"; u = sorted(s.astype(str).unique())
            rango = " | ".join(u) if len(u) <= 8 else f"{len(u)} categorías"
        filas.append([c, tipo, DESC.get(c, ""), rango, int(s.isna().sum()), int(s.nunique())])
    return pd.DataFrame(filas, columns=["variable", "tipo", "descripcion", "rango_o_categorias", "nulos", "valores_unicos"])

diccionario = reg_tabla("diccionario_datos", construir_diccionario(df))
reg_tabla("dataset_modelado", df)

# =============================================================================
# BLOQUE 2. EDA
# =============================================================================
log("BLOQUE 2: EDA")
COLOR_E = {1: "#c0392b", 2: "#e6a817", 3: "#2471a3"}
COLOR_T = {"No": "#c0392b", "Tal vez": "#e6a817", "Sí": "#1e8449"}
NUM_EDA = ["num_personas", "ingreso_mensual_cop", "area_vivienda_m2", "area_cubierta_m2", "consumo_kwh_mes", "demanda_maxima_kw",
           "factor_carga", "costo_factura_mes_cop", "generacion_potencial_kwp", "inversion_estimada_cop", "ahorro_anual_cop",
           "porcentaje_ahorro", "retorno_simple_anios", "vpn_10_anios_cop", "tir_porcentaje", "carga_familiar",
           "ratio_consumo_ingreso_pct", "indice_eficiencia_vivienda", "cobertura_solar_pct", "ratio_diurno_6_17",
           "ratio_noche_18_22", "disposicion_pago_mensual_cop"]
CAT_EDA = ["estrato", "zona", "tipo_vivienda", "nivel_educativo", "ocupacion_principal", "tipo_cubierta", "orientacion_cubierta",
           "sombras_presentes", "tiene_internet", "tiene_smartphone", "nivel_apropiacion_tecnologica", "dispositivo_ia_interes",
           "principal_barrera", "principal_motivacion", "prioridad_gobierno", "fuente_financiacion", "dispuesto_instalar_paneles"]

desc_num = df[NUM_EDA].describe(percentiles=[.25, .5, .75]).T
desc_num["asimetria"] = df[NUM_EDA].skew(); desc_num["curtosis"] = df[NUM_EDA].kurt()
reg_tabla("estadisticas_descriptivas_modelo", desc_num.round(3), index=True)
reg_tabla("resumen_por_estrato", df.groupby("estrato")[["consumo_kwh_mes", "ingreso_mensual_cop", "area_cubierta_m2", "generacion_potencial_kwp",
                                                          "inversion_estimada_cop", "ahorro_anual_cop", "retorno_simple_anios"]].mean().round(2), index=True)

def fig_univariado(var):
    if var in NUM_EDA:
        fig = make_subplots(rows=1, cols=2, column_widths=[0.7, 0.3], subplot_titles=("Distribución", "Boxplot por estrato"))
        fig.add_trace(go.Histogram(x=df[var], nbinsx=40, marker_color="#2471a3", name=var), row=1, col=1)
        for s in (1, 2, 3):
            fig.add_trace(go.Box(y=df.loc[df.estrato == s, var], name=f"E{s}", marker_color=COLOR_E[s], showlegend=False), row=1, col=2)
        fig.update_layout(title=f"{var}  (media={df[var].mean():.2f} | mediana={df[var].median():.2f} | asimetría={df[var].skew():.2f})", height=470)
    else:
        vc = df[var].astype(str).value_counts().reset_index(); vc.columns = [var, "hogares"]
        fig = px.bar(vc, x=var, y="hogares", text="hogares", color=var, title=f"Frecuencias de {var}")
        fig.update_layout(showlegend=False, height=470)
    return fig

for v in ["consumo_kwh_mes", "ingreso_mensual_cop", "retorno_simple_anios", "dispuesto_instalar_paneles"]:
    reg_fig(f"eda_univariado_{v}", fig_univariado(v), clave=(v == "consumo_kwh_mes"))

def fig_bivariado(x, y, color):
    col = None if color == "(ninguno)" else color
    num_x, num_y = x in NUM_EDA, y in NUM_EDA
    if num_x and num_y:
        try:
            fig = px.scatter(df, x=x, y=y, color=col if col is None else df[col].astype(str), trendline="ols", opacity=0.65,
                             title=f"{y} vs {x}")
        except Exception:
            fig = px.scatter(df, x=x, y=y, color=col if col is None else df[col].astype(str), opacity=0.65, title=f"{y} vs {x}")
        r, p = stats.pearsonr(df[x], df[y]); rho, _ = stats.spearmanr(df[x], df[y])
        fig.update_layout(title=f"{y} vs {x}  |  Pearson r={r:.2f}  Spearman ρ={rho:.2f}")
    elif (not num_x) and num_y:
        fig = px.box(df, x=x, y=y, color=x, points="outliers", title=f"{y} según {x}")
        grupos = [g[y].values for _, g in df.groupby(x)]
        if len(grupos) > 1:
            h, p = stats.kruskal(*grupos)
            fig.update_layout(title=f"{y} según {x}  |  Kruskal-Wallis H={h:.1f}, p={p:.2g}")
    elif num_x and (not num_y):
        fig = px.box(df, x=y, y=x, color=y, points="outliers", title=f"{x} según {y}")
    else:
        ct = pd.crosstab(df[x], df[y], normalize="index") * 100
        fig = px.imshow(ct.round(1), text_auto=True, aspect="auto", color_continuous_scale="Blues", title=f"% fila: {y} dentro de cada {x}")
        chi2, p, _, _ = stats.chi2_contingency(pd.crosstab(df[x], df[y]))
        fig.update_layout(title=f"{y} dentro de cada {x}  |  χ²={chi2:.1f}, p={p:.2g}")
    fig.update_layout(height=520)
    return fig

reg_fig("eda_bivariado_consumo_ingreso", fig_bivariado("ingreso_mensual_cop", "consumo_kwh_mes", "estrato"), clave=True)
reg_fig("eda_bivariado_inversion_ahorro", fig_bivariado("inversion_estimada_cop", "ahorro_anual_cop", "estrato"))

EDA_MV = [c for c in NUM_EDA if c not in ("disposicion_pago_mensual_cop",)]
X_mv = StandardScaler().fit_transform(df[EDA_MV])
pca_mv = PCA(n_components=min(10, X_mv.shape[1]), random_state=SEED).fit(X_mv)
Z = pca_mv.transform(X_mv)
var_exp = pca_mv.explained_variance_ratio_
fig_var = go.Figure()
fig_var.add_bar(x=[f"PC{i + 1}" for i in range(len(var_exp))], y=var_exp * 100, name="Varianza explicada")
fig_var.add_scatter(x=[f"PC{i + 1}" for i in range(len(var_exp))], y=np.cumsum(var_exp) * 100, name="Acumulada", mode="lines+markers")
fig_var.update_layout(title="PCA: varianza explicada (%)", yaxis_title="%", height=420)
reg_fig("pca_varianza", fig_var)
cargas = pd.DataFrame(pca_mv.components_[:3].T, index=EDA_MV, columns=["PC1", "PC2", "PC3"]).round(3)
reg_tabla("pca_cargas", cargas, index=True)

dfZ = pd.DataFrame({"PC1": Z[:, 0], "PC2": Z[:, 1], "PC3": Z[:, 2], "estrato": df["estrato"].astype(str),
                    "adopcion": df["dispuesto_instalar_paneles"], "zona": df["zona"]})
fig_pca2 = px.scatter(dfZ, x="PC1", y="PC2", color="adopcion", symbol="estrato", color_discrete_map=COLOR_T, opacity=0.75,
                      title=f"PCA 2D (PC1 {var_exp[0]*100:.1f}% + PC2 {var_exp[1]*100:.1f}%)")
fig_pca2.update_layout(height=520)
fig_pca3 = px.scatter_3d(dfZ, x="PC1", y="PC2", z="PC3", color="adopcion", symbol="estrato", color_discrete_map=COLOR_T, opacity=0.75,
                         title=f"PCA 3D (varianza acumulada {var_exp[:3].sum()*100:.1f}%)")
fig_pca3.update_layout(height=620)
reg_fig("pca_2d", fig_pca2, clave=True); reg_fig("pca_3d", fig_pca3)

tsne = TSNE(n_components=2, perplexity=30, init="pca", learning_rate="auto", random_state=SEED)
Zt = tsne.fit_transform(X_mv)
dfT = pd.DataFrame({"tSNE1": Zt[:, 0], "tSNE2": Zt[:, 1], "estrato": df["estrato"].astype(str), "adopcion": df["dispuesto_instalar_paneles"]})
fig_tsne = px.scatter(dfT, x="tSNE1", y="tSNE2", color="adopcion", symbol="estrato", color_discrete_map=COLOR_T, opacity=0.75,
                      title="t-SNE (perplexity=30)")
fig_tsne.update_layout(height=520)
reg_fig("tsne_2d", fig_tsne, clave=True)

_rng = np.random.default_rng(SEED)
idx_dend = np.sort(_rng.choice(len(df), 90, replace=False))
fig_dend0 = ff.create_dendrogram(X_mv[idx_dend], labels=[f"H{df.id_hogar.iloc[i][1:]}-E{df.estrato.iloc[i]}" for i in idx_dend],
                                 linkagefun=lambda x: linkage(x, "ward"))
fig_dend0.update_layout(title="Dendrograma (Ward) de 90 hogares", height=560, xaxis_tickangle=90)
reg_fig("dendrograma_eda", fig_dend0)

corr = df[NUM_EDA].corr(method="spearman").round(2)
fig_corr = px.imshow(corr, text_auto=False, zmin=-1, zmax=1, color_continuous_scale="RdBu_r", aspect="auto",
                     title="Matriz de correlación de Spearman")
fig_corr.update_layout(height=780)
reg_fig("matriz_correlacion", fig_corr, clave=True)
reg_tabla("correlaciones_spearman", corr, index=True)

tab_t = pd.crosstab(df["estrato"], df["dispuesto_instalar_paneles"], normalize="index")[CLASES] * 100
fig_t1 = px.bar(tab_t.reset_index().melt(id_vars="estrato", var_name="respuesta", value_name="porcentaje"), x="estrato", y="porcentaje",
                color="respuesta", color_discrete_map=COLOR_T, barmode="stack", text_auto=".1f")
tab_z = pd.crosstab(df["zona"], df["dispuesto_instalar_paneles"], normalize="index")[CLASES] * 100
fig_t2 = px.bar(tab_z.reset_index().melt(id_vars="zona", var_name="respuesta", value_name="porcentaje"), x="zona", y="porcentaje",
                color="respuesta", color_discrete_map=COLOR_T, barmode="stack", text_auto=".1f")
fig_t3 = px.box(df, x="dispuesto_instalar_paneles", y="retorno_simple_anios", color="dispuesto_instalar_paneles", color_discrete_map=COLOR_T,
                category_orders={"dispuesto_instalar_paneles": CLASES})
fig_target = make_subplots(rows=1, cols=3, subplot_titles=("Por estrato (%)", "Por zona (%)", "Retorno simple"))
for tr in fig_t1.data: fig_target.add_trace(tr, row=1, col=1)
for tr in fig_t2.data:
    tr.showlegend = False; fig_target.add_trace(tr, row=1, col=2)
for tr in fig_t3.data:
    tr.showlegend = False; fig_target.add_trace(tr, row=1, col=3)
fig_target.update_layout(barmode="stack", height=480, title="Análisis del target")
reg_fig("analisis_target", fig_target, clave=True)
bal = df["dispuesto_instalar_paneles"].value_counts(normalize=True).reindex(CLASES) * 100
print("Balance del target (%):", bal.round(1).to_dict())

def fig_pareto(poblacion="No + Tal vez", nivel="Barrera"):
    d = df
    if poblacion == "Solo No": d = df[df.dispuesto_instalar_paneles == "No"]
    elif poblacion == "No + Tal vez": d = df[df.dispuesto_instalar_paneles != "Sí"]
    if nivel == "Barrera":
        s = d["principal_barrera"].value_counts(); ttl = "Pareto barreras"
    else:
        s = (d["estrato"].map(lambda e: f"E{e}") + " | " + d["zona"] + " | " + d["principal_barrera"]).value_counts()
        ttl = "Pareto segmentos"
    s = s.sort_values(ascending=False); acum = s.cumsum() / s.sum() * 100
    n20 = max(1, int(math.ceil(0.2 * len(s)))); pct20 = acum.iloc[n20 - 1]; n80 = int((acum < 80).sum() + 1)
    fig = make_subplots(specs=[[{"secondary_y": True}]])
    colores = ["#c0392b" if i < n20 else "#aab7b8" for i in range(len(s))]
    fig.add_bar(x=s.index, y=s.values, marker_color=colores, name="Hogares", secondary_y=False)
    fig.add_scatter(x=s.index, y=acum.values, mode="lines+markers", name="% acumulado", line=dict(color="#2471a3"), secondary_y=True)
    fig.add_hline(y=80, line_dash="dash", line_color="gray", secondary_y=True)
    fig.update_yaxes(title_text="Hogares", secondary_y=False); fig.update_yaxes(title_text="% acumulado", range=[0, 105], secondary_y=True)
    fig.update_layout(title=f"{ttl} (n={len(d)}). 20% superior explica {pct20:.1f}%; {n80} para 80%", height=520, xaxis_tickangle=-45, title_font_size=13)
    tabla = pd.DataFrame({"categoria": s.index, "hogares": s.values, "pct": (s / s.sum() * 100).round(1).values, "pct_acumulado": acum.round(1).values})
    return fig, tabla

fig_par, tabla_par = fig_pareto("No + Tal vez", "Barrera")
fig_par2, tabla_par2 = fig_pareto("No + Tal vez", "Segmentos")
reg_fig("pareto_barreras", fig_par, clave=True); reg_fig("pareto_segmentos", fig_par2)
reg_tabla("pareto_barreras", tabla_par); reg_tabla("pareto_segmentos", tabla_par2)
print("Pareto:\n", tabla_par.to_string(index=False))
log("EDA completado")

# =============================================================================
# BLOQUE 3. CLASIFICACIÓN: 6 clásicos + RED NEURONAL MULTIMODAL PyTorch
# =============================================================================
log("BLOQUE 3: Clasificación de adopción (clásicos + NN multimodal)")

NUM_CLS = ["estrato", "num_personas", "num_menores", "num_adultos_mayores", "ingreso_mensual_cop", "consumo_kwh_mes", "tarifa_kwh_cop",
           "area_vivienda_m2", "area_cubierta_m2", "num_paneles_estimados", "potencia_panel_w", "porcentaje_ahorro", "retorno_simple_anios",
           "carga_familiar", "ratio_consumo_ingreso_pct", "kwh_por_persona", "kwh_por_m2", "indice_eficiencia_vivienda",
           "cobertura_solar_pct", "brecha_digital", "kwp_calc"]
CAT_CLS = ["zona", "nivel_educativo", "ocupacion_principal", "tiene_empresa", "tipo_cubierta", "sombras_presentes", "tiene_internet",
           "tiene_smartphone", "tiene_medidor_inteligente", "recibe_subsidio", "conoce_reciclaje_paneles", "interes_comunidad_energetica"]
HORA_FEATS = HORAS
FEATS_CLS = NUM_CLS + CAT_CLS
FEATS_CLS_FULL = FEATS_CLS + HORA_FEATS

X_all = df[FEATS_CLS_FULL].copy()
y_all = df["dispuesto_instalar_paneles"].map({c: i for i, c in enumerate(CLASES)}).astype(int).values
X_tr, X_te, y_tr, y_te = train_test_split(X_all, y_all, test_size=0.20, stratify=y_all, random_state=SEED)
idx_te = X_te.index
print(f"Entrenamiento: {len(X_tr)} | Prueba: {len(X_te)} | Tabulares: {len(FEATS_CLS)} | Horarias: {len(HORA_FEATS)} | Total: {X_all.shape[1]}")

if NN_OK:
    class HogarAdopcionNet(nn.Module):
        def __init__(self, n_num, cat_card, n_horas=24, n_classes=3, emb_max=16, dropout=0.3):
            super().__init__()
            self.mlp_num = nn.Sequential(
                nn.Linear(n_num, 128), nn.BatchNorm1d(128), nn.GELU(), nn.Dropout(0.2),
                nn.Linear(128, 64), nn.GELU(), nn.LayerNorm(64)
            )
            self.emb_dims = {c: min(emb_max, max(2, (card + 1) // 2)) for c, card in cat_card.items()}
            self.embs = nn.ModuleDict({c: nn.Embedding(card + 1, self.emb_dims[c], padding_idx=0) for c, card in cat_card.items()})
            total_emb = sum(self.emb_dims.values())
            self.cat_mlp = nn.Sequential(nn.Linear(total_emb, 64), nn.GELU(), nn.Dropout(0.2))
            self.conv = nn.Sequential(
                nn.Conv1d(1, 16, kernel_size=3, padding=1), nn.BatchNorm1d(16), nn.GELU(),
                nn.Conv1d(16, 32, kernel_size=3, padding=1), nn.BatchNorm1d(32), nn.GELU(),
                nn.AdaptiveAvgPool1d(1), nn.Flatten()
            )
            self.head_hora = nn.Sequential(nn.Linear(32, 32), nn.GELU())
            self.head = nn.Sequential(
                nn.Linear(64 + 64 + 32, 128), nn.GELU(), nn.Dropout(dropout),
                nn.Linear(128, n_classes)
            )
            self.cat_cols_ordered = list(cat_card.keys())

        def forward(self, x_num, x_cat, x_hora):
            a = self.mlp_num(x_num)
            embs = [self.embs[c](x_cat[:, i]) for i, c in enumerate(self.cat_cols_ordered)]
            b = self.cat_mlp(torch.cat(embs, dim=1))
            c = self.head_hora(self.conv(x_hora.unsqueeze(1)))
            return self.head(torch.cat([a, b, c], dim=1))

    class TorchTabularClassifier(BaseEstimator, ClassifierMixin):
        _estimator_type = "classifier"

        def __sklearn_tags__(self):
            tags = super().__sklearn_tags__()
            tags.estimator_type = "classifier"
            return tags

        def __init__(self, num_cols, cat_cols, hora_cols, epochs=40, batch_size=64, lr=3e-4,
                     weight_decay=1e-4, patience=8, dropout=0.3, seed=SEED, device=None):
            self.num_cols = num_cols
            self.cat_cols = cat_cols
            self.hora_cols = hora_cols
            self.epochs = epochs
            self.batch_size = batch_size
            self.lr = lr
            self.weight_decay = weight_decay
            self.patience = patience
            self.dropout = dropout
            self.seed = seed
            self.device = str(device) if device is not None else None

        def _prep_fit(self, X):
            self.scaler_num_ = StandardScaler().fit(X[self.num_cols].values)
            self.enc_cat_ = {}
            self.cat_card_ = {}
            for c in self.cat_cols:
                le = LabelEncoder().fit(X[c].astype(str).values)
                self.enc_cat_[c] = le
                self.cat_card_[c] = len(le.classes_)
            H = X[self.hora_cols].values.astype(float)
            self.scaler_hora_ = StandardScaler().fit(H / np.clip(H.sum(1, keepdims=True), 1e-9, None))
            return self

        def _transform(self, X):
            xn = self.scaler_num_.transform(X[self.num_cols].values).astype(np.float32)
            xc = np.zeros((len(X), len(self.cat_cols)), dtype=np.int64)
            for i, c in enumerate(self.cat_cols):
                le = self.enc_cat_[c]
                mapeado = {cl: j + 1 for j, cl in enumerate(le.classes_)}
                xc[:, i] = [mapeado.get(v, 0) for v in X[c].astype(str).values]
            H = X[self.hora_cols].values.astype(float)
            Hn = H / np.clip(H.sum(1, keepdims=True), 1e-9, None)
            xh = self.scaler_hora_.transform(Hn).astype(np.float32)
            return xn, xc, xh

        def fit(self, X, y):
            X = X.reset_index(drop=True)
            y = np.asarray(y).astype(int)
            self._prep_fit(X)
            xn, xc, xh = self._transform(X)
            n_tr = int(0.85 * len(X))
            idx = np.arange(len(X))
            rng = np.random.default_rng(self.seed)
            rng.shuffle(idx)
            tr, va = idx[:n_tr], idx[n_tr:]
            dev = torch.device(self.device) if self.device else (
                torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
            )
            self.device_ = str(dev)
            self.model_ = HogarAdopcionNet(
                n_num=xn.shape[1],
                cat_card={c: self.cat_card_[c] for c in self.cat_cols},
                n_horas=xh.shape[1], n_classes=3, dropout=self.dropout
            ).to(dev)
            self.model_.cat_cols_ordered = self.cat_cols
            cnt = np.bincount(y, minlength=3).astype(float)
            w = (len(y) / (3 * np.clip(cnt, 1, None)))
            crit = nn.CrossEntropyLoss(weight=torch.tensor(w, dtype=torch.float32, device=dev))
            opt = torch.optim.AdamW(self.model_.parameters(), lr=self.lr, weight_decay=self.weight_decay)
            sched = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(opt, T_0=10, T_mult=2)
            Xn_t = torch.tensor(xn, device=dev)
            Xc_t = torch.tensor(xc, device=dev)
            Xh_t = torch.tensor(xh, device=dev)
            y_t = torch.tensor(y, device=dev)
            best_f1, best_state, bad = -1.0, None, 0
            self.history_ = {"epoch": [], "loss_tr": [], "loss_va": [], "f1_va": []}
            for ep in range(self.epochs):
                self.model_.train()
                perm = torch.randperm(n_tr, device=dev)
                tot_loss = 0.0
                for i in range(0, n_tr, self.batch_size):
                    b = perm[i:i + self.batch_size]
                    gidx = torch.tensor(tr, device=dev)[b]
                    logits = self.model_(Xn_t[gidx], Xc_t[gidx], Xh_t[gidx])
                    loss = crit(logits, y_t[gidx])
                    opt.zero_grad(); loss.backward(); opt.step()
                    tot_loss += loss.item() * len(b)
                sched.step()
                self.model_.eval()
                with torch.no_grad():
                    gv = torch.tensor(va, device=dev)
                    logits = self.model_(Xn_t[gv], Xc_t[gv], Xh_t[gv])
                    loss_v = crit(logits, y_t[gv]).item()
                    pv = torch.softmax(logits, 1).cpu().numpy().argmax(1)
                f1v = f1_score(y[va], pv, average="macro")
                self.history_["epoch"].append(ep + 1)
                self.history_["loss_tr"].append(tot_loss / n_tr)
                self.history_["loss_va"].append(loss_v)
                self.history_["f1_va"].append(f1v)
                if f1v > best_f1 + 1e-4:
                    best_f1 = f1v
                    bad = 0
                    best_state = {k: v.detach().cpu().clone() for k, v in self.model_.state_dict().items()}
                else:
                    bad += 1
                    if bad >= self.patience:
                        break
            if best_state is not None:
                self.model_.load_state_dict(best_state)
            self.best_f1_ = best_f1
            self.classes_ = np.array([0, 1, 2])
            return self

        def predict_proba(self, X):
            from sklearn.utils.validation import check_is_fitted
            check_is_fitted(self, ["model_"])
            X = X.reset_index(drop=True)
            xn, xc, xh = self._transform(X)
            dev = torch.device(self.device_) if self.device_ else (
                torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
            )
            self.model_.eval()
            with torch.no_grad():
                logits = self.model_(
                    torch.tensor(xn, device=dev),
                    torch.tensor(xc, device=dev),
                    torch.tensor(xh, device=dev),
                )
                return torch.softmax(logits, 1).cpu().numpy()

        def predict(self, X):
            return self.predict_proba(X).argmax(1)


class _MLP_Horas(BaseEstimator, ClassifierMixin):
    _estimator_type = "classifier"
    def __init__(self, num_cols, cat_cols, hora_cols, seed=SEED):
        self.num_cols = num_cols; self.cat_cols = cat_cols; self.hora_cols = hora_cols; self.seed = seed
    def fit(self, X, y):
        self.pre_ = ColumnTransformer([
            ("num", StandardScaler(), self.num_cols),
            ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), self.cat_cols),
            ("hora", StandardScaler(), self.hora_cols)], verbose_feature_names_out=True).fit(X)
        Xp = self.pre_.transform(X)
        self.clf_ = MLPClassifier(hidden_layer_sizes=(256, 128, 64), activation="relu", alpha=1e-3,
                                   learning_rate_init=1e-3, early_stopping=True, validation_fraction=0.15,
                                   n_iter_no_change=25, max_iter=1500, random_state=self.seed).fit(Xp, y)
        self.classes_ = self.clf_.classes_
        return self
    def predict_proba(self, X): return self.clf_.predict_proba(self.pre_.transform(X))
    def predict(self, X): return self.clf_.predict(X)


def nuevo_preproceso(num=NUM_CLS, cat=CAT_CLS):
    return ColumnTransformer([("num", StandardScaler(), num),
                              ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat)], verbose_feature_names_out=True)

def _sel_cls(X):
    return X[FEATS_CLS]

def nuevo_pipeline_clasico(modelo):
    return Pipeline([
        ("sel", FunctionTransformer(_sel_cls, validate=False)),
        ("prep", nuevo_preproceso()),
        ("clf", modelo),
    ])

CV5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

def afinar(nombre, modelo, espacio, n_iter=10):
    pipe = nuevo_pipeline_clasico(modelo)
    rs = RandomizedSearchCV(pipe, espacio, n_iter=n_iter, cv=StratifiedKFold(3, shuffle=True, random_state=SEED),
                            scoring="f1_macro", random_state=SEED, n_jobs=1, refit=False)
    rs.fit(X_tr, y_tr)
    mejores = {k.replace("clf__", ""): v for k, v in rs.best_params_.items()}
    log(f"  ajuste {nombre}: f1_macro CV3={rs.best_score_:.3f}")
    return mejores

lgb_base = dict(random_state=SEED, n_jobs=-1, verbose=-1)
p_lgb = afinar("LightGBM", LGBMClassifier(**lgb_base),
               {"clf__n_estimators": [150, 250, 400], "clf__num_leaves": [7, 15, 31], "clf__learning_rate": [0.03, 0.05, 0.1],
                "clf__subsample": [0.7, 0.85, 1.0], "clf__colsample_bytree": [0.6, 0.8, 1.0], "clf__min_child_samples": [10, 20, 30],
                "clf__subsample_freq": [1]})
p_rf = afinar("Random Forest", RandomForestClassifier(random_state=SEED, n_jobs=-1, class_weight="balanced_subsample"),
              {"clf__n_estimators": [300, 500], "clf__max_depth": [None, 6, 10], "clf__min_samples_leaf": [1, 2, 4],
               "clf__max_features": ["sqrt", 0.5]}, n_iter=8)

def modelos_clasificacion():
    lr = LogisticRegression(max_iter=3000, C=1.0, random_state=SEED)
    rf = RandomForestClassifier(random_state=SEED, n_jobs=-1, class_weight="balanced_subsample", **p_rf)
    lg = LGBMClassifier(**{**lgb_base, **p_lgb})
    sv = SVC(kernel="rbf", C=2.0, gamma="scale", probability=True, random_state=SEED)
    ml = MLPClassifier(hidden_layer_sizes=(128, 64, 32), activation="relu", alpha=1e-3, learning_rate_init=1e-3,
                       early_stopping=True, validation_fraction=0.15, n_iter_no_change=25, max_iter=1000, random_state=SEED)
    st = StackingClassifier(estimators=[("lr", clone(lr)), ("rf", clone(rf)), ("lgbm", clone(lg)), ("svm", clone(sv)),
                                        ("mlp", clone(ml))],
                            final_estimator=LogisticRegression(max_iter=3000, random_state=SEED), cv=5, stack_method="predict_proba", n_jobs=1)
    return {"Regresión Logística": lr, "Random Forest": rf, "LightGBM": lg, "SVM (RBF)": sv,
            "MLP (sklearn 128-64-32)": ml, "Stacking": st}

PIPES = {n: nuevo_pipeline_clasico(m) for n, m in modelos_clasificacion().items()}
if NN_OK:
    PIPES["Red Neuronal Multimodal (PyTorch)"] = TorchTabularClassifier(
        num_cols=NUM_CLS, cat_cols=CAT_CLS, hora_cols=HORA_FEATS,
        epochs=40, batch_size=64, lr=3e-4, weight_decay=1e-4, patience=8, dropout=0.3, seed=SEED
    )
else:
    PIPES["Red Neuronal Multimodal (PyTorch)"] = _MLP_Horas(NUM_CLS, CAT_CLS, HORA_FEATS)


def brier_multiclase(y, P):
    Y = np.eye(P.shape[1])[y]
    return float(np.mean(np.sum((P - Y) ** 2, axis=1)))

def ece_multiclase(y, P, bins=10):
    conf = P.max(axis=1); pred = P.argmax(axis=1); ok = (pred == y).astype(float)
    edges = np.linspace(0, 1, bins + 1); e = 0.0
    for i in range(bins):
        m = (conf > edges[i]) & (conf <= edges[i + 1])
        if m.any():
            e += m.mean() * abs(ok[m].mean() - conf[m].mean())
    return float(e)

def metricas_clf(y, P):
    pred = P.argmax(axis=1); Y = np.eye(P.shape[1])[y]
    return {"accuracy": accuracy_score(y, pred), "balanced_acc": balanced_accuracy_score(y, pred), "f1_macro": f1_score(y, pred, average="macro"),
            "f1_weighted": f1_score(y, pred, average="weighted"), "roc_auc_ovr": roc_auc_score(y, P, multi_class="ovr", average="macro"),
            "pr_auc_macro": float(np.mean([average_precision_score(Y[:, k], P[:, k]) for k in range(P.shape[1])])),
            "mcc": matthews_corrcoef(y, pred), "kappa": cohen_kappa_score(y, pred), "brier": brier_multiclase(y, P),
            "log_loss": log_loss(y, P, labels=[0, 1, 2]), "ece": ece_multiclase(y, P)}

filas_cv, filas_te, PROBA_TE = [], [], {}
scoring_cv = {"accuracy": "accuracy", "f1_macro": "f1_macro", "balanced_acc": "balanced_accuracy", "roc_auc_ovr": "roc_auc_ovr", "neg_log_loss": "neg_log_loss"}
for nombre, pipe in PIPES.items():
    t1 = time.time()
    cvres = cross_validate(pipe, X_tr, y_tr, cv=CV5, scoring=scoring_cv, n_jobs=1, error_score="raise")
    pipe.fit(X_tr, y_tr)
    P = pipe.predict_proba(X_te); PROBA_TE[nombre] = P
    m = metricas_clf(y_te, P)
    filas_cv.append({"modelo": nombre, **{f"cv_{k}": cvres[f"test_{k}"].mean() for k in scoring_cv}, "cv_f1_macro_std": cvres["test_f1_macro"].std()})
    filas_te.append({"modelo": nombre, **m})
    log(f"  {nombre:34s} CV f1_macro={cvres['test_f1_macro'].mean():.3f} | test f1_macro={m['f1_macro']:.3f} AUC={m['roc_auc_ovr']:.3f} ({time.time() - t1:.0f}s)")
tab_cv = pd.DataFrame(filas_cv).set_index("modelo"); tab_te = pd.DataFrame(filas_te).set_index("modelo")
tab_clf = tab_cv.join(tab_te, how="left").round(4).sort_values("cv_f1_macro", ascending=False)
reg_tabla("clasificacion_comparacion_modelos", tab_clf, index=True)
MEJOR = tab_clf.index[0]
print("MEJOR MODELO (por f1_macro en CV):", MEJOR)
MEJOR_PIPE = PIPES[MEJOR]; P_best = PROBA_TE[MEJOR]; pred_best = P_best.argmax(axis=1)
assert "Red Neuronal Multimodal (PyTorch)" in tab_clf.index, "La NN no entró en la comparación"

if NN_OK and isinstance(PIPES["Red Neuronal Multimodal (PyTorch)"], TorchTabularClassifier):
    hist = PIPES["Red Neuronal Multimodal (PyTorch)"].history_
    fig_nn = make_subplots(rows=1, cols=2, subplot_titles=("Pérdida", "F1-macro validación"))
    fig_nn.add_trace(go.Scatter(x=hist["epoch"], y=hist["loss_tr"], mode="lines+markers", name="Train loss", line=dict(color="#2471a3")), row=1, col=1)
    fig_nn.add_trace(go.Scatter(x=hist["epoch"], y=hist["loss_va"], mode="lines+markers", name="Val loss", line=dict(color="#c0392b")), row=1, col=1)
    fig_nn.add_trace(go.Scatter(x=hist["epoch"], y=hist["f1_va"], mode="lines+markers", name="F1 val", line=dict(color="#1e8449")), row=1, col=2)
    fig_nn.update_layout(height=440, title=f"Curvas NN (mejor F1 val={PIPES['Red Neuronal Multimodal (PyTorch)'].best_f1_:.3f})")
    reg_fig("nn_curvas_entrenamiento", fig_nn, clave=True)

met_plot = ["accuracy", "balanced_acc", "f1_macro", "roc_auc_ovr", "pr_auc_macro", "mcc", "kappa"]
dfp = tab_te[met_plot].reset_index().melt(id_vars="modelo", var_name="métrica", value_name="valor")
fig_cmp = px.bar(dfp, x="modelo", y="valor", color="métrica", barmode="group", title="Comparación de modelos (test 20%)")
fig_cmp.update_layout(height=520, xaxis_tickangle=-20)
reg_fig("clf_comparacion_modelos", fig_cmp, clave=True)
fig_cv = go.Figure()
fig_cv.add_bar(x=tab_cv.index, y=tab_cv["cv_f1_macro"], error_y=dict(type="data", array=tab_cv["cv_f1_macro_std"]), marker_color="#2471a3", name="f1_macro CV")
fig_cv.update_layout(title="F1-macro en CV estratificada (media ± desv.)", height=420, yaxis_range=[0, 1])
reg_fig("clf_cv_f1", fig_cv)

cm = confusion_matrix(y_te, pred_best, labels=[0, 1, 2]); cmn = cm / cm.sum(axis=1, keepdims=True)
fig_cm = make_subplots(rows=1, cols=2, subplot_titles=("Conteos", "Normalizada"))
fig_cm.add_trace(go.Heatmap(z=cm, x=CLASES, y=CLASES, text=cm, texttemplate="%{text}", colorscale="Blues", showscale=False), row=1, col=1)
fig_cm.add_trace(go.Heatmap(z=cmn, x=CLASES, y=CLASES, text=np.round(cmn, 2), texttemplate="%{text}", colorscale="Greens", showscale=False), row=1, col=2)
fig_cm.update_xaxes(title_text="Predicho"); fig_cm.update_yaxes(title_text="Real", autorange="reversed")
fig_cm.update_layout(title=f"Matriz de confusión - {MEJOR}", height=450)
reg_fig("clf_matriz_confusion", fig_cm, clave=True)

def curvas_roc_pr(y, P, nombre):
    Y = np.eye(3)[y]
    fig = make_subplots(rows=1, cols=2, subplot_titles=("ROC", "Precisión-Recall"))
    for k, c in enumerate(CLASES):
        fpr, tpr, _ = roc_curve(Y[:, k], P[:, k]); auc = roc_auc_score(Y[:, k], P[:, k])
        fig.add_trace(go.Scatter(x=fpr, y=tpr, mode="lines", name=f"{c} (AUC={auc:.3f})", line=dict(color=COLOR_T[c])), row=1, col=1)
        pr, rc, _ = precision_recall_curve(Y[:, k], P[:, k]); ap = average_precision_score(Y[:, k], P[:, k])
        fig.add_trace(go.Scatter(x=rc, y=pr, mode="lines", name=f"{c} (AP={ap:.3f})", line=dict(color=COLOR_T[c], dash="dot")), row=1, col=2)
    fig.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode="lines", line=dict(color="gray", dash="dash"), showlegend=False), row=1, col=1)
    fig.update_layout(title=f"{nombre}: AUC macro={roc_auc_score(y, P, multi_class='ovr'):.3f}", height=470)
    return fig
reg_fig("clf_roc_pr", curvas_roc_pr(y_te, P_best, MEJOR), clave=True)

cal_iso = CalibratedClassifierCV(estimator=clone(MEJOR_PIPE), method="isotonic", cv=5)
cal_iso.fit(X_tr, y_tr)
P_cal = cal_iso.predict_proba(X_te)
m_ant = metricas_clf(y_te, P_best); m_des = metricas_clf(y_te, P_cal)
tab_cal = pd.DataFrame({"antes": m_ant, "después": m_des}).loc[["brier", "log_loss", "ece", "accuracy", "f1_macro", "roc_auc_ovr"]].round(4)
reg_tabla("clasificacion_calibracion", tab_cal, index=True)
USAR_CAL = m_des["brier"] < m_ant["brier"]
MODELO_FINAL_CLF = cal_iso if USAR_CAL else MEJOR_PIPE
print("Modelo final:", MEJOR + (" + calibración" if USAR_CAL else ""))

fig_cal = make_subplots(rows=1, cols=3, subplot_titles=[f"Clase '{c}'" for c in CLASES], shared_yaxes=True)
Yte = np.eye(3)[y_te]
for k, c in enumerate(CLASES):
    for P_, nm, col in ((P_best, "Sin calibrar", "#c0392b"), (P_cal, "Isotónica", "#1e8449")):
        try:
            pt, pp = calibration_curve(Yte[:, k], P_[:, k], n_bins=6, strategy="quantile")
            fig_cal.add_trace(go.Scatter(x=pp, y=pt, mode="lines+markers", name=nm, line=dict(color=col), showlegend=(k == 0)), row=1, col=k + 1)
        except Exception:
            pass
    fig_cal.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode="lines", line=dict(color="gray", dash="dash"), showlegend=False), row=1, col=k + 1)
fig_cal.update_layout(title=f"Calibración - {MEJOR}  |  Brier: {m_ant['brier']:.3f} -> {m_des['brier']:.3f}", height=460)
reg_fig("clf_calibracion", fig_cal, clave=True)

# Permutation importance
X_eval_pi = X_te[FEATS_CLS_FULL] if isinstance(MODELO_FINAL_CLF, (TorchTabularClassifier, _MLP_Horas)) else X_te[FEATS_CLS]
pi = permutation_importance(MODELO_FINAL_CLF, X_eval_pi, y_te, scoring="f1_macro", n_repeats=15, random_state=SEED, n_jobs=1)
tab_pi = pd.DataFrame({"variable": list(X_eval_pi.columns), "importancia": pi.importances_mean, "desv": pi.importances_std}).sort_values("importancia", ascending=False)
reg_tabla("clasificacion_permutation_importance", tab_pi.round(4))
top_pi = tab_pi.head(15).iloc[::-1]
fig_pi = go.Figure(go.Bar(x=top_pi["importancia"], y=top_pi["variable"], orientation="h", error_x=dict(type="data", array=top_pi["desv"]), marker_color="#2471a3"))
fig_pi.update_layout(title="Permutation Importance (caída F1-macro, test)", height=520)
reg_fig("clf_permutation_importance", fig_pi, clave=True)

# SHAP protegido
arboles = [n for n in ("Random Forest", "LightGBM") if n in tab_cv.index]
ARBOL = max(arboles, key=lambda n: tab_cv.loc[n, "cv_f1_macro"])
SHAP_OK = False
ORIG_VARS, SV_O, base_vals = [], np.zeros((len(X_te), 0, 3)), np.zeros(3)
try:
    arbol_pipe = PIPES[ARBOL]
    prep_a = arbol_pipe.named_steps["prep"]
    Xt_te = prep_a.transform(X_te[FEATS_CLS]); nombres_ohe = [re.sub(r"^(num|cat)__", "", n) for n in prep_a.get_feature_names_out()]
    def _orig(n):
        for c in CAT_CLS:
            if n.startswith(c + "_"): return c
        return n
    orig_de = np.array([_orig(n) for n in nombres_ohe])
    expl = shap.TreeExplainer(arbol_pipe.named_steps["clf"])
    sv_raw = expl.shap_values(Xt_te)
    def _a3d(sv, n, f, k=3):
        if isinstance(sv, list): a = np.stack(sv, axis=2)
        else:
            a = np.asarray(sv)
            if a.ndim == 2: a = a[:, :, None]
            if a.shape[0] == k and a.shape[1] == n: a = np.transpose(a, (1, 2, 0))
        return a
    SV = _a3d(sv_raw, Xt_te.shape[0], Xt_te.shape[1])
    base_vals = np.atleast_1d(expl.expected_value)
    ORIG_VARS = list(dict.fromkeys(orig_de))
    SV_O = np.stack([SV[:, orig_de == v, :].sum(axis=1) for v in ORIG_VARS], axis=1)
    imp_cls = np.abs(SV_O).mean(axis=0)
    tab_shap = pd.DataFrame(imp_cls, index=ORIG_VARS, columns=[f"SHAP|{c}|" for c in CLASES])
    tab_shap["SHAP_medio_abs_total"] = imp_cls.sum(axis=1)
    tab_shap = tab_shap.sort_values("SHAP_medio_abs_total", ascending=False)
    reg_tabla("clasificacion_shap_importancia", tab_shap.round(4), index=True)
    top = tab_shap.head(15).iloc[::-1]
    fig_shap_bar = go.Figure()
    for k, c in enumerate(CLASES):
        fig_shap_bar.add_bar(y=top.index, x=top[f"SHAP|{c}|"], orientation="h", name=c, marker_color=COLOR_T[c])
    fig_shap_bar.update_layout(barmode="stack", title=f"SHAP - importancia media ({ARBOL})", height=560)
    reg_fig("clf_shap_resumen", fig_shap_bar, clave=True)
    SHAP_OK = True
except Exception as _e_shap:
    print("  [SHAP] no disponible:", str(_e_shap)[:120])

def fig_shap_beeswarm(clase="Sí", top_n=12):
    if not SHAP_OK:
        return go.Figure().update_layout(title="SHAP no disponible")
    k = CLASES.index(clase)
    orden = np.argsort(-np.abs(SV_O[:, :, k]).mean(axis=0))[:top_n][::-1]
    fig = go.Figure(); rng_ = np.random.default_rng(SEED)
    for pos, j in enumerate(orden):
        v = ORIG_VARS[j]; col = X_te[v]
        val = col.astype("category").cat.codes.astype(float).values if not pd.api.types.is_numeric_dtype(col) else col.values.astype(float)
        rng_v = (val - val.min()) / (np.ptp(val) + 1e-9)
        fig.add_trace(go.Scatter(x=SV_O[:, j, k], y=pos + rng_.uniform(-0.28, 0.28, len(val)), mode="markers",
                                 marker=dict(size=5, color=rng_v, colorscale="RdBu_r", showscale=(pos == len(orden) - 1)),
                                 text=[f"{v}={x}" for x in col.values], hoverinfo="text+x", showlegend=False))
    fig.update_yaxes(tickvals=list(range(len(orden))), ticktext=[ORIG_VARS[j] for j in orden])
    fig.update_layout(title=f"SHAP beeswarm - clase '{clase}' ({ARBOL})", xaxis_title="SHAP", height=560)
    return fig
reg_fig("clf_shap_beeswarm_si", fig_shap_beeswarm("Sí"))

def fig_shap_force(i=0, clase="Sí", top_n=10):
    if not SHAP_OK:
        return go.Figure().update_layout(title="SHAP no disponible")
    k = CLASES.index(clase); i = int(np.clip(i, 0, SV_O.shape[0] - 1))
    ph = SV_O[i, :, k]; orden = np.argsort(-np.abs(ph))[:top_n][::-1]
    base = float(base_vals[k]) if len(base_vals) > k else 0.0
    etiquetas = [f"{ORIG_VARS[j]} = {X_te.iloc[i][ORIG_VARS[j]]}" for j in orden]
    fig = go.Figure(go.Waterfall(orientation="h", y=etiquetas, x=[ph[j] for j in orden], measure=["relative"] * len(orden),
                                 increasing=dict(marker_color="#1e8449"), decreasing=dict(marker_color="#c0392b")))
    real = CLASES[int(y_te[i])]; pred = CLASES[int(P_best[i].argmax())]
    fig.update_layout(title=f"SHAP waterfall hogar {X_te.index[i]} | clase={clase} | real={real} | predicho={pred}", height=520, title_font_size=13)
    return fig
reg_fig("clf_shap_force_ejemplo", fig_shap_force(0, "Sí"))

# Fairness
P_oof = cross_val_predict(clone(MEJOR_PIPE), X_all, y_all, cv=CV5, method="predict_proba", n_jobs=1)
pred_oof = P_oof.argmax(axis=1)
df_f = df[["estrato", "zona", "jefe_hogar_genero"]].copy(); df_f["estrato"] = "E" + df_f["estrato"].astype(str)
df_f["estrato_zona"] = df_f["estrato"] + " | " + df_f["zona"]
df_f["real"] = y_all; df_f["pred"] = pred_oof; df_f["p_si"] = P_oof[:, 2]
def tabla_fairness(col):
    filas = []
    for g, d in df_f.groupby(col):
        yr, yp = d["real"].values, d["pred"].values
        pos_real, pos_pred = (yr == 2), (yp == 2)
        tpr = pos_pred[pos_real].mean() if pos_real.any() else np.nan
        fpr = pos_pred[~pos_real].mean() if (~pos_real).any() else np.nan
        filas.append({"grupo": g, "n": len(d), "tasa_real_Si": pos_real.mean(), "tasa_pred_Si": pos_pred.mean(), "TPR_Si": tpr, "FPR_Si": fpr,
                      "exactitud": (yr == yp).mean(), "f1_macro": f1_score(yr, yp, average="macro")})
    t = pd.DataFrame(filas).set_index("grupo")
    resumen = {"dif_paridad_demografica": t["tasa_pred_Si"].max() - t["tasa_pred_Si"].min(),
               "dif_oportunidad_igual": t["TPR_Si"].max() - t["TPR_Si"].min(),
               "impacto_dispar": t["tasa_pred_Si"].min() / max(t["tasa_pred_Si"].max(), 1e-9),
               "dif_exactitud": t["exactitud"].max() - t["exactitud"].min()}
    return t.round(3), {k: round(float(v), 3) for k, v in resumen.items()}
FAIR = {}
for col in ["estrato", "zona", "estrato_zona", "jefe_hogar_genero"]:
    FAIR[col] = tabla_fairness(col)
    reg_tabla(f"fairness_{col}", FAIR[col][0], index=True)
def fig_fairness(col):
    t, r = FAIR[col]
    fig = make_subplots(rows=1, cols=2, subplot_titles=("Tasa real vs predicha 'Sí'", "TPR / FPR / Exactitud"))
    fig.add_bar(x=t.index, y=t["tasa_real_Si"], name="Real", marker_color="#aab7b8", row=1, col=1)
    fig.add_bar(x=t.index, y=t["tasa_pred_Si"], name="Predicha", marker_color="#2471a3", row=1, col=1)
    fig.add_bar(x=t.index, y=t["TPR_Si"], name="TPR", marker_color="#1e8449", row=1, col=2)
    fig.add_bar(x=t.index, y=t["FPR_Si"], name="FPR", marker_color="#c0392b", row=1, col=2)
    fig.add_bar(x=t.index, y=t["exactitud"], name="Exactitud", marker_color="#e6a817", row=1, col=2)
    alerta = " ⚠ impacto dispar < 0.8" if r["impacto_dispar"] < 0.8 else ""
    fig.update_layout(barmode="group", height=470, title=f"Equidad por {col} | Δparidad={r['dif_paridad_demografica']}{alerta}", title_font_size=13)
    return fig
reg_fig("fairness_estrato", fig_fairness("estrato"), clave=True); reg_fig("fairness_zona", fig_fairness("zona"))
print("Fairness estrato:", FAIR["estrato"][1])
log("Clasificación completada")

# =============================================================================
# BLOQUE 4. REGRESIÓN
# =============================================================================
log("BLOQUE 4: Regresión")
NUM_REG = ["estrato", "num_personas", "consumo_kwh_mes", "tarifa_kwh_cop", "area_cubierta_m2", "inclinacion_cubierta_grados",
           "num_paneles_estimados", "potencia_panel_w", "ratio_diurno_6_17"]
CAT_REG = ["zona", "tipo_cubierta", "orientacion_cubierta", "sombras_presentes"]
FEATS_REG = NUM_REG + CAT_REG
TARGETS_REG = {"ahorro_anual_cop": "Ahorro anual (COP)", "retorno_simple_anios": "Retorno simple (años)"}
Xr = df[FEATS_REG].copy()
KF5 = KFold(5, shuffle=True, random_state=SEED)

class TorchRegressor(BaseEstimator):
    _estimator_type = "regressor"

    def __init__(self, hidden=(128, 64, 32), epochs=200, batch_size=64, lr=1e-3, patience=20, seed=SEED):
        self.hidden = hidden; self.epochs = epochs; self.batch_size = batch_size
        self.lr = lr; self.patience = patience; self.seed = seed

    def fit(self, X, y):
        X = np.asarray(X, dtype=np.float32); y = np.asarray(y, dtype=np.float32).reshape(-1, 1)
        torch.manual_seed(self.seed)
        layers = []; prev = X.shape[1]
        for h in self.hidden:
            layers += [nn.Linear(prev, h), nn.GELU(), nn.Dropout(0.15)]; prev = h
        layers += [nn.Linear(prev, 1)]
        self.net_ = nn.Sequential(*layers)
        dev = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
        self.net_.to(dev)
        opt = torch.optim.AdamW(self.net_.parameters(), lr=self.lr, weight_decay=1e-4)
        crit = nn.MSELoss()
        Xt = torch.tensor(X, device=dev); yt = torch.tensor(y, device=dev)
        idx = np.arange(len(X)); rng = np.random.default_rng(self.seed); rng.shuffle(idx)
        n_tr = int(0.85 * len(X)); tr, va = idx[:n_tr], idx[n_tr:]
        best_loss, bad, best_state = np.inf, 0, None
        for ep in range(self.epochs):
            self.net_.train()
            perm = torch.tensor(rng.permutation(n_tr), device=dev)
            for i in range(0, n_tr, self.batch_size):
                b = perm[i:i + self.batch_size]
                gidx = torch.tensor(tr, device=dev)[b]
                loss = crit(self.net_(Xt[gidx]), yt[gidx])
                opt.zero_grad(); loss.backward(); opt.step()
            self.net_.eval()
            with torch.no_grad():
                gv = torch.tensor(va, device=dev)
                loss_v = crit(self.net_(Xt[gv]), yt[gv]).item()
            if loss_v < best_loss - 1e-4:
                best_loss = loss_v; bad = 0
                best_state = {k: v.detach().cpu().clone() for k, v in self.net_.state_dict().items()}
            else:
                bad += 1
                if bad >= self.patience: break
        if best_state is not None: self.net_.load_state_dict(best_state)
        return self

    def predict(self, X):
        dev = next(self.net_.parameters()).device
        self.net_.eval()
        with torch.no_grad():
            Xt = torch.tensor(np.asarray(X, dtype=np.float32), device=dev)
            return self.net_(Xt).cpu().numpy().ravel()

REG = {}
for tgt in TARGETS_REG:
    yr = df[tgt].values
    Xr_tr, Xr_te, yr_tr, yr_te_ = train_test_split(Xr, yr, test_size=0.2, random_state=SEED, stratify=df["estrato"])
    modelos = {
        "Ridge (línea base)": Ridge(alpha=1.0),
        "Random Forest": RandomForestRegressor(n_estimators=500, min_samples_leaf=2, n_jobs=-1, random_state=SEED),
        "MLP (sklearn 64-32)": TransformedTargetRegressor(MLPRegressor(hidden_layer_sizes=(64, 32), alpha=1e-2, early_stopping=True, max_iter=2000, random_state=SEED), transformer=StandardScaler()),
    }
    if NN_OK:
        modelos["Red Neuronal (PyTorch 128-64-32)"] = TorchRegressor(hidden=(128, 64, 32), epochs=200, batch_size=64, lr=1e-3, patience=20, seed=SEED)
    filas, fitted, preds = [], {}, {}
    for nombre, mdl in modelos.items():
        pipe = Pipeline([("prep", nuevo_preproceso(NUM_REG, CAT_REG)), ("reg", mdl)])
        cv = cross_validate(pipe, Xr_tr, yr_tr, cv=KF5, scoring={"r2": "r2", "mae": "neg_mean_absolute_error", "rmse": "neg_root_mean_squared_error"}, n_jobs=1)
        pipe.fit(Xr_tr, yr_tr); p = pipe.predict(Xr_te)
        filas.append({"modelo": nombre, "cv_R2": cv["test_r2"].mean(), "cv_MAE": -cv["test_mae"].mean(), "cv_RMSE": -cv["test_rmse"].mean(),
                      "test_R2": r2_score(yr_te_, p), "test_MAE": mean_absolute_error(yr_te_, p), "test_RMSE": float(np.sqrt(mean_squared_error(yr_te_, p)))})
        fitted[nombre] = pipe; preds[nombre] = p
    tab = pd.DataFrame(filas).set_index("modelo").round(4).sort_values("cv_R2", ascending=False)
    reg_tabla(f"regresion_{tgt}", tab, index=True)
    mejor = tab.index[0]
    final = clone(fitted[mejor]).fit(Xr, yr)
    REG[tgt] = dict(tabla=tab, mejor=mejor, modelo=final, y_te=yr_te_, pred=preds[mejor], X_te=Xr_te, fitted=fitted)
    log(f"  {TARGETS_REG[tgt]}: mejor={mejor} | CV R²={tab.loc[mejor, 'cv_R2']:.3f} | test R²={tab.loc[mejor, 'test_R2']:.3f}")
    pr = preds[mejor]; res = yr_te_ - pr
    fig = make_subplots(rows=2, cols=2, subplot_titles=("Real vs predicho", "Residuos", "Histograma residuos", "QQ-plot"))
    lim = [min(yr_te_.min(), pr.min()), max(yr_te_.max(), pr.max())]
    fig.add_trace(go.Scatter(x=yr_te_, y=pr, mode="markers", marker=dict(color=Xr_te["estrato"].map(COLOR_E), size=6, opacity=0.7), showlegend=False), row=1, col=1)
    fig.add_trace(go.Scatter(x=lim, y=lim, mode="lines", line=dict(color="gray", dash="dash"), showlegend=False), row=1, col=1)
    fig.add_trace(go.Scatter(x=pr, y=res, mode="markers", marker=dict(size=6, opacity=0.7, color="#2471a3"), showlegend=False), row=1, col=2)
    fig.add_hline(y=0, line_dash="dash", line_color="gray", row=1, col=2)
    fig.add_trace(go.Histogram(x=res, nbinsx=30, marker_color="#8e44ad", showlegend=False), row=2, col=1)
    (osm, osr), (slope, intercept, _) = stats.probplot(res, dist="norm")
    fig.add_trace(go.Scatter(x=osm, y=osr, mode="markers", marker=dict(size=5, color="#1e8449"), showlegend=False), row=2, col=2)
    fig.add_trace(go.Scatter(x=osm, y=slope * np.asarray(osm) + intercept, mode="lines", line=dict(color="gray", dash="dash"), showlegend=False), row=2, col=2)
    fig.update_layout(height=760, title=f"{TARGETS_REG[tgt]} - {mejor} | R²={r2_score(yr_te_, pr):.3f}", title_font_size=13)
    reg_fig(f"reg_diagnostico_{tgt}", fig, clave=True)
    imp = permutation_importance(fitted[mejor], Xr_te, yr_te_, scoring="r2", n_repeats=10, random_state=SEED, n_jobs=1)
    REG[tgt]["importancia"] = pd.DataFrame({"variable": FEATS_REG, "importancia": imp.importances_mean}).sort_values("importancia", ascending=False)
    reg_tabla(f"regresion_importancia_{tgt}", REG[tgt]["importancia"].round(4))
fig_cmp_reg = go.Figure()
for tgt in TARGETS_REG:
    t = REG[tgt]["tabla"]
    fig_cmp_reg.add_bar(x=t.index, y=t["test_R2"], name=TARGETS_REG[tgt])
fig_cmp_reg.update_layout(barmode="group", title="R² test por modelo y objetivo", height=420, yaxis_range=[0, 1.05])
reg_fig("reg_comparacion", fig_cmp_reg)
COSTO_KWP_REF = float(df["costo_kwp_cop"].median())
log("Regresión completada")

# =============================================================================
# BLOQUE 5. CLUSTERING
# =============================================================================
log("BLOQUE 5: Clustering")
CL_FEATS = ["consumo_kwh_mes", "ingreso_mensual_cop", "num_personas", "area_cubierta_m2", "generacion_potencial_kwp", "porcentaje_ahorro",
            "retorno_simple_anios", "ratio_diurno_6_17", "ratio_noche_18_22", "indice_eficiencia_vivienda", "carga_familiar", "estrato"]
sc_cl = StandardScaler().fit(df[CL_FEATS]); Xc = sc_cl.transform(df[CL_FEATS])
pca_cl = PCA(n_components=2, random_state=SEED).fit(Xc); Zc = pca_cl.transform(Xc)
filas_k = []
for k in range(2, 11):
    km = KMeans(n_clusters=k, n_init=20, random_state=SEED).fit(Xc)
    filas_k.append({"k": k, "inercia": km.inertia_, "silhouette": silhouette_score(Xc, km.labels_),
                    "calinski_harabasz": calinski_harabasz_score(Xc, km.labels_), "davies_bouldin": davies_bouldin_score(Xc, km.labels_)})
tab_k = pd.DataFrame(filas_k).round(4); reg_tabla("kmeans_seleccion_k", tab_k)
K_OPT = int(tab_k[(tab_k.k >= 3) & (tab_k.k <= 7)].sort_values("silhouette", ascending=False).iloc[0]["k"])
print(f"K óptimo: {K_OPT}")
fig_k = make_subplots(rows=1, cols=3, subplot_titles=("Codo", "Silhouette", "Davies-Bouldin"))
fig_k.add_trace(go.Scatter(x=tab_k.k, y=tab_k.inercia, mode="lines+markers", showlegend=False), row=1, col=1)
fig_k.add_trace(go.Scatter(x=tab_k.k, y=tab_k.silhouette, mode="lines+markers", marker_color="#1e8449", showlegend=False), row=1, col=2)
fig_k.add_trace(go.Scatter(x=tab_k.k, y=tab_k.davies_bouldin, mode="lines+markers", marker_color="#c0392b", showlegend=False), row=1, col=3)
for c in (1, 2, 3): fig_k.add_vline(x=K_OPT, line_dash="dash", line_color="gray", row=1, col=c)
fig_k.update_layout(title=f"Selección de k (k={K_OPT})", height=420)
reg_fig("kmeans_seleccion_k", fig_k, clave=True)
KM = KMeans(n_clusters=K_OPT, n_init=30, random_state=SEED).fit(Xc)
df["cluster_kmeans"] = KM.labels_
medias = df.groupby("cluster_kmeans")[["consumo_kwh_mes", "estrato", "ingreso_mensual_cop"]].mean()
orden_cons = medias["consumo_kwh_mes"].rank(method="first").astype(int)
def _nombre_cluster(c):
    r = int(orden_cons[c]); n = len(medias)
    nivel = "consumo bajo" if r <= n / 3 else ("consumo medio" if r <= 2 * n / 3 else "consumo alto")
    return f"C{c}: {nivel}, estrato≈{medias.loc[c, 'estrato']:.1f}"
NOMBRES_CL = {c: _nombre_cluster(c) for c in medias.index}
df["segmento"] = df["cluster_kmeans"].map(NOMBRES_CL)
perfil = df.groupby("segmento").agg(hogares=("id_hogar", "size"), consumo_kwh_mes=("consumo_kwh_mes", "mean"), ingreso_cop=("ingreso_mensual_cop", "mean"),
                                    personas=("num_personas", "mean"), area_cubierta_m2=("area_cubierta_m2", "mean"), kwp=("generacion_potencial_kwp", "mean"),
                                    pct_ahorro=("porcentaje_ahorro", "mean"), retorno_anios=("retorno_simple_anios", "mean"),
                                    ratio_diurno=("ratio_diurno_6_17", "mean"), pct_rural=("zona", lambda s: 100 * (s == "Rural").mean()),
                                    pct_dispuestos_Si=("dispuesto_instalar_paneles", lambda s: 100 * (s == "Sí").mean()),
                                    vpn_positivo_pct=("vpn_10_anios_cop", lambda s: 100 * (s > 0).mean())).round(2)
reg_tabla("kmeans_perfil_segmentos", perfil, index=True)
print(perfil.to_string())
fig_clu = px.scatter(x=Zc[:, 0], y=Zc[:, 1], color=df["segmento"], opacity=0.75, title=f"K-Means k={K_OPT} en PCA 2D")
cen2 = pca_cl.transform(KM.cluster_centers_)
fig_clu.add_trace(go.Scatter(x=cen2[:, 0], y=cen2[:, 1], mode="markers+text", text=[f"C{i}" for i in range(K_OPT)], marker=dict(symbol="x", size=14, color="black"), name="Centroides"))
fig_clu.update_layout(height=560)
reg_fig("kmeans_pca", fig_clu, clave=True)
zc = pd.DataFrame(StandardScaler().fit_transform(df[CL_FEATS]), columns=CL_FEATS).groupby(df["segmento"].values).mean().round(2)
fig_heat = px.imshow(zc.T, color_continuous_scale="RdBu_r", zmin=-2, zmax=2, text_auto=True, aspect="auto", title="Perfil z-score por segmento")
fig_heat.update_layout(height=520); reg_fig("kmeans_perfil_heatmap", fig_heat)
horas_seg = df.groupby("segmento")[HORAS].mean().T * 1000
fig_hseg = px.line(horas_seg, labels={"index": "Hora", "value": "Wh/hora", "segmento": "Segmento"}, title="Perfil horario por segmento")
fig_hseg.update_layout(height=450); reg_fig("kmeans_perfil_horario", fig_hseg)
AGG = AgglomerativeClustering(n_clusters=K_OPT, linkage="ward").fit(Xc)
df["cluster_jerarquico"] = AGG.labels_
ARI = adjusted_rand_score(df["cluster_kmeans"], df["cluster_jerarquico"])
sil_h = silhouette_score(Xc, AGG.labels_)
idx_h = np.sort(np.random.default_rng(SEED).choice(len(df), 120, replace=False))
fig_dend = ff.create_dendrogram(Xc[idx_h], labels=[f"E{df.estrato.iloc[i]}-{df.id_hogar.iloc[i][1:]}" for i in idx_h], linkagefun=lambda x: linkage(x, "ward"))
fig_dend.update_layout(title=f"Dendrograma Ward (120 hogares) | k={K_OPT}, silhouette={sil_h:.3f}, ARI={ARI:.2f}", height=620, xaxis_tickangle=90)
reg_fig("dendrograma_clustering", fig_dend, clave=True)
pca_db = PCA(n_components=0.90, random_state=SEED).fit(Xc); Zdb = pca_db.transform(Xc)
MIN_S = 8
kd = np.sort(NearestNeighbors(n_neighbors=MIN_S).fit(Zdb).kneighbors(Zdb)[0][:, -1])
mejor_db = None
for q in (0.90, 0.92, 0.94, 0.96, 0.97, 0.98, 0.99):
    eps = float(np.quantile(kd, q)); lab = DBSCAN(eps=eps, min_samples=MIN_S).fit_predict(Zdb)
    frac = float((lab == -1).mean())
    if 0.015 <= frac <= 0.10:
        mejor_db = (q, eps, lab, frac); break
if mejor_db is None:
    eps = float(np.quantile(kd, 0.95)); lab = DBSCAN(eps=eps, min_samples=MIN_S).fit_predict(Zdb); mejor_db = (0.95, eps, lab, float((lab == -1).mean()))
q_db, EPS_DB, lab_db, frac_db = mejor_db
df["dbscan"] = lab_db; df["atipico_dbscan"] = (lab_db == -1)
fig_kd = go.Figure(go.Scatter(y=kd, mode="lines")); fig_kd.add_hline(y=EPS_DB, line_dash="dash", line_color="red", annotation_text=f"eps={EPS_DB:.2f}")
fig_kd.update_layout(title=f"k-distancia (k={MIN_S}) - criterio eps", height=380); reg_fig("dbscan_kdistancia", fig_kd)
dbp = pd.DataFrame({"PC1": Zc[:, 0], "PC2": Zc[:, 1], "tipo": np.where(lab_db == -1, "Atípico", "Normal"), "estrato": df["estrato"].astype(str), "id": df["id_hogar"]})
fig_db = px.scatter(dbp, x="PC1", y="PC2", color="tipo", symbol="estrato", color_discrete_map={"Atípico": "#c0392b", "Normal": "#aab7b8"}, hover_name="id", opacity=0.8,
                    title=f"DBSCAN: {int((lab_db==-1).sum())} hogares atípicos")
fig_db.update_layout(height=520); reg_fig("dbscan_atipicos", fig_db, clave=True)
atip = df.loc[df.atipico_dbscan, ["id_hogar", "estrato", "zona", "num_personas", "consumo_kwh_mes", "ingreso_mensual_cop", "generacion_potencial_kwp",
                                  "porcentaje_ahorro", "retorno_simple_anios", "dispuesto_instalar_paneles"]].round(2)
reg_tabla("dbscan_hogares_atipicos", atip)
reg_tabla("hogares_con_segmentos", df[["id_hogar", "estrato", "zona", "cluster_kmeans", "segmento", "cluster_jerarquico", "dbscan", "atipico_dbscan"]])
log("Clustering completado")

# =============================================================================
# BLOQUE 6. SERIES TEMPORALES
# =============================================================================
log("BLOQUE 6: Series temporales")
FECHA_INI, FECHA_FIN = "2024-11-01", "2026-10-31"
FECHA_CORTE_BT = "2026-04-30"
H_FINAL = 365
def generar_series_sinteticas(df_, semilla=SEED):
    rg = np.random.default_rng(semilla)
    fechas = pd.date_range(FECHA_INI, FECHA_FIN, freq="D"); n = len(fechas); doy = fechas.dayofyear.values
    cons0 = float(df_["consumo_kwh_dia_promedio"].mean()); kwp0 = float(df_["generacion_potencial_kwp"].mean())
    hsp0 = float(df_["horas_sol_pico"].mean()); amp_est = float(np.clip(df_["variacion_estacional_pct"].abs().mean() / 100, 0.02, 0.12))
    lluvia = np.cos(4 * np.pi * (doy - 120) / 365.0)
    navidad = np.exp(-0.5 * ((((doy - 355) + 182) % 365 - 182) / 10.0) ** 2)
    findes = np.where(fechas.dayofweek >= 5, 1.06, 1.0)
    tend = 1 + 0.03 * np.arange(n) / 365.0
    ar = np.zeros(n)
    for t in range(1, n): ar[t] = 0.55 * ar[t - 1] + rg.normal(0, 0.028)
    consumo = cons0 * (1 + amp_est * lluvia + 0.07 * navidad) * findes * tend * np.exp(ar)
    etiqueta = np.zeros(n, dtype=int)
    idx_an = rg.choice(np.arange(20, n - 20), 12, replace=False)
    for j, t in enumerate(idx_an):
        consumo[t] *= rg.uniform(1.45, 1.9) if j % 4 != 3 else rg.uniform(0.45, 0.6); etiqueta[t] = 1
    nubes = np.clip(1 - 0.10 * (lluvia > 0.2) * lluvia + rg.normal(0, 0.12, n), 0.30, 1.20)
    gen = kwp0 * hsp0 * 0.78 * (1 - 0.10 * np.clip(lluvia, 0, 1)) * np.clip(nubes, 0.3, 1.2) * (1 - 0.005 * np.arange(n) / 365.0)
    return pd.DataFrame({"consumo_kwh_dia": consumo, "generacion_kwh_dia": gen, "anomalia_real": etiqueta}, index=fechas)
serie_d = generar_series_sinteticas(df)
serie_d["balance_kwh_dia"] = serie_d["generacion_kwh_dia"] - serie_d["consumo_kwh_dia"]
serie_m = serie_d[["consumo_kwh_dia", "generacion_kwh_dia"]].resample("MS").sum().rename(columns={"consumo_kwh_dia": "consumo_kwh_mes", "generacion_kwh_dia": "generacion_kwh_mes"})
serie_m["autosuficiencia_pct"] = 100 * serie_m["generacion_kwh_mes"] / serie_m["consumo_kwh_mes"]
reg_tabla("serie_diaria_sintetica", serie_d.round(4), index=True); reg_tabla("serie_mensual_sintetica", serie_m.round(3), index=True)
SERIES = {"Consumo": ("consumo_kwh_dia", "consumo_kwh_mes"), "Generación FV": ("generacion_kwh_dia", "generacion_kwh_mes")}
fig_ts = make_subplots(rows=2, cols=1, shared_xaxes=True, subplot_titles=("Diaria", "Mensual"))
fig_ts.add_trace(go.Scatter(x=serie_d.index, y=serie_d["consumo_kwh_dia"], name="Consumo", line=dict(color="#c0392b", width=1)), row=1, col=1)
fig_ts.add_trace(go.Scatter(x=serie_d.index, y=serie_d["generacion_kwh_dia"], name="Generación", line=dict(color="#e6a817", width=1)), row=1, col=1)
fig_ts.add_trace(go.Scatter(x=serie_m.index, y=serie_m["consumo_kwh_mes"], name="Consumo (mes)", mode="lines+markers", line=dict(color="#c0392b")), row=2, col=1)
fig_ts.add_trace(go.Scatter(x=serie_m.index, y=serie_m["generacion_kwh_mes"], name="Generación (mes)", mode="lines+markers", line=dict(color="#e6a817")), row=2, col=1)
fig_ts.update_layout(height=620, title="Series sintéticas")
reg_fig("ts_series", fig_ts, clave=True)
def fig_stl(serie="Consumo", frecuencia="Diaria"):
    cd, cm_ = SERIES[serie]
    if frecuencia == "Diaria":
        y = serie_d[cd]; per = 7; res = STL(y, period=per, robust=True).fit(); tit = "STL diaria"
    else:
        y = serie_m[cm_]; per = 12; res = STL(y, period=per, robust=True).fit(); tit = "STL mensual"
    fig = make_subplots(rows=4, cols=1, shared_xaxes=True, subplot_titles=("Observada", "Tendencia", "Estacionalidad", "Residuo"), vertical_spacing=0.06)
    for i, (comp, col) in enumerate(((y, "#2471a3"), (res.trend, "#1e8449"), (res.seasonal, "#8e44ad"), (res.resid, "#c0392b")), start=1):
        fig.add_trace(go.Scatter(x=y.index, y=comp, mode="lines", line=dict(color=col, width=1.3), showlegend=False), row=i, col=1)
    fe = max(0.0, 1 - res.resid.var() / (res.resid + res.seasonal).var()); ft = max(0.0, 1 - res.resid.var() / (res.resid + res.trend).var())
    fig.update_layout(height=760, title=f"{serie} - {tit} | F_est={fe:.2f}, F_tend={ft:.2f}")
    return fig
reg_fig("ts_stl_consumo_diaria", fig_stl("Consumo", "Diaria"), clave=True); reg_fig("ts_stl_consumo_mensual", fig_stl("Consumo", "Mensual"))
def fig_acf_pacf(serie="Consumo", frecuencia="Diaria"):
    cd, cm_ = SERIES[serie]
    if frecuencia == "Diaria":
        y = serie_d[cd].diff().dropna(); nl = 40; tit = "diaria, d=1"
    else:
        y = serie_m[cm_]; nl = 10; tit = "mensual"
    a = acf(y, nlags=nl, fft=True); p = pacf(y, nlags=nl, method="ywm"); ci = 1.96 / np.sqrt(len(y))
    fig = make_subplots(rows=1, cols=2, subplot_titles=("ACF", "PACF"))
    for c, v in ((1, a), (2, p)):
        fig.add_trace(go.Bar(x=list(range(len(v))), y=v, marker_color="#2471a3", showlegend=False), row=1, col=c)
        fig.add_hline(y=ci, line_dash="dash", line_color="red", row=1, col=c); fig.add_hline(y=-ci, line_dash="dash", line_color="red", row=1, col=c)
    fig.update_layout(height=430, title=f"{serie}: ACF/PACF ({tit})")
    return fig
reg_fig("ts_acf_pacf_consumo", fig_acf_pacf("Consumo", "Diaria"), clave=True)
def fourier(index, K=3):
    doy = index.dayofyear.values; cols = {}
    for k in range(1, K + 1):
        cols[f"s{k}"] = np.sin(2 * np.pi * k * doy / 365.25); cols[f"c{k}"] = np.cos(2 * np.pi * k * doy / 365.25)
    return pd.DataFrame(cols, index=index)
def ajustar_sarimax(y):
    Xf = fourier(y.index)
    for orden, sord in (((1, 0, 1), (1, 0, 1, 7)), ((1, 0, 0), (1, 0, 0, 7)), ((1, 0, 0), (0, 0, 0, 0))):
        try:
            m = SARIMAX(y, exog=Xf, order=orden, seasonal_order=sord, trend="c", enforce_stationarity=False, enforce_invertibility=False)
            return m.fit(disp=False, maxiter=200), orden, sord
        except Exception:
            continue
    raise RuntimeError("SARIMAX no convergió")
def pronostico_sarimax(res, y_index_fin, h, n_sim=300):
    fidx = pd.date_range(y_index_fin + pd.Timedelta(days=1), periods=h, freq="D"); Xf = fourier(fidx)
    mean = res.get_forecast(steps=h, exog=Xf).predicted_mean.values
    try:
        sim = np.asarray(res.simulate(nsimulations=h, repetitions=n_sim, anchor="end", exog=Xf)).reshape(h, -1)
    except Exception:
        sd = float(np.std(res.resid[10:])); sim = mean[:, None] + np.random.default_rng(SEED).normal(0, sd, (h, n_sim))
    return fidx, mean, sim
def ajustar_prophet(y):
    if not PROPHET_OK: return None
    d = pd.DataFrame({"ds": y.index, "y": y.values})
    m = Prophet(yearly_seasonality=4, weekly_seasonality=True, daily_seasonality=False, seasonality_mode="multiplicative", interval_width=0.90,
                changepoint_prior_scale=0.05, uncertainty_samples=300)
    m.fit(d); return m
def pronostico_prophet(m, h):
    fut = m.make_future_dataframe(periods=h, freq="D"); fc = m.predict(fut).iloc[-h:]
    try:
        samp = np.asarray(m.predictive_samples(fut)["yhat"])[-h:, :]
    except Exception:
        samp = None
    return pd.DatetimeIndex(fc["ds"]), fc["yhat"].values, samp
def a_mensual(idx, valores): return pd.Series(valores, index=idx).resample("MS").sum()
def a_mensual_bandas(idx, sims, lo=5, hi=95):
    if sims is None: return None, None
    df_s = pd.DataFrame(sims, index=idx).resample("MS").sum()
    return np.percentile(df_s.values, lo, axis=1), np.percentile(df_s.values, hi, axis=1)
def evaluar(real_m, pred_m):
    ok = real_m.index.intersection(pred_m.index)
    r, p = real_m.loc[ok].values, pred_m.loc[ok].values
    return {"MAE_mensual": mean_absolute_error(r, p), "RMSE_mensual": float(np.sqrt(mean_squared_error(r, p))), "MAPE_%": float(np.mean(np.abs((r - p) / r)) * 100)}
TS = {}
filas_bt = []
for nombre, (cd, cm_) in SERIES.items():
    ys = serie_d[cd]; ts_train = ys[:FECHA_CORTE_BT]; ts_test = ys[FECHA_CORTE_BT:].iloc[1:]
    real_m = ts_test.resample("MS").sum()
    ym = serie_d[cd].resample("MS").sum(); base_m = pd.Series([ym.get(i - pd.DateOffset(years=1), np.nan) for i in real_m.index], index=real_m.index)
    filas_bt.append({"serie": nombre, "modelo": "Naive estacional", **evaluar(real_m, base_m.dropna())})
    t1 = time.time(); res_bt, o_, so_ = ajustar_sarimax(ts_train)
    fidx, mean, sim = pronostico_sarimax(res_bt, ts_train.index[-1], len(ts_test), n_sim=100)
    filas_bt.append({"serie": nombre, "modelo": f"SARIMAX{o_}x{so_}+Fourier", **evaluar(real_m, a_mensual(fidx, mean))})
    if PROPHET_OK:
        mp = ajustar_prophet(ts_train); pidx, pm, _ = pronostico_prophet(mp, len(ts_test))
        filas_bt.append({"serie": nombre, "modelo": "Prophet", **evaluar(real_m, a_mensual(pidx, pm))})
    res_f, o_, so_ = ajustar_sarimax(ys)
    fidx, mean, sim = pronostico_sarimax(res_f, ys.index[-1], H_FINAL, n_sim=300)
    m_sar = a_mensual(fidx, mean); lo_s, hi_s = a_mensual_bandas(fidx, sim)
    pro = None
    if PROPHET_OK:
        mp_f = ajustar_prophet(ys); pidx, pm, psamp = pronostico_prophet(mp_f, H_FINAL)
        m_pro = a_mensual(pidx, pm); lo_p, hi_p = a_mensual_bandas(pidx, psamp)
        pro = dict(modelo=mp_f, serie_mensual=m_pro, lo=lo_p, hi=hi_p, diario=pd.Series(pm, index=pidx))
    TS[nombre] = dict(sarimax=res_f, orden=(o_, so_), sar_mensual=m_sar, sar_lo=lo_s, sar_hi=hi_s, prophet=pro, hist_m=ym, sar_diario=pd.Series(mean, index=fidx))
    log(f"  {nombre}: SARIMAX{o_}x{so_} y Prophet ajustados ({time.time()-t1:.0f}s)")
tab_bt = pd.DataFrame(filas_bt).round(3); reg_tabla("ts_backtest_6_meses", tab_bt)
def fig_pronostico(serie="Consumo"):
    d = TS[serie]; fig = go.Figure()
    fig.add_trace(go.Scatter(x=d["hist_m"].index, y=d["hist_m"].values, mode="lines+markers", name="Histórico", line=dict(color="black")))
    if d["sar_lo"] is not None:
        fig.add_trace(go.Scatter(x=list(d["sar_mensual"].index) + list(d["sar_mensual"].index[::-1]), y=list(d["sar_hi"]) + list(d["sar_lo"][::-1]),
                                 fill="toself", fillcolor="rgba(36,113,163,0.15)", line=dict(width=0), name="SARIMAX IC 90%", hoverinfo="skip"))
    fig.add_trace(go.Scatter(x=d["sar_mensual"].index, y=d["sar_mensual"].values, mode="lines+markers", name="SARIMAX", line=dict(color="#2471a3")))
    if d["prophet"] is not None:
        p = d["prophet"]
        fig.add_trace(go.Scatter(x=p["serie_mensual"].index, y=p["serie_mensual"].values, mode="lines+markers", name="Prophet", line=dict(color="#c0392b", dash="dash")))
    fig.update_layout(title=f"Pronóstico 12 meses: {serie}", height=520, yaxis_title="kWh/mes", hovermode="x unified")
    return fig
reg_fig("ts_pronostico_consumo", fig_pronostico("Consumo"), clave=True); reg_fig("ts_pronostico_generacion", fig_pronostico("Generación FV"))
filas_pron = []
for nm, d in TS.items():
    base = pd.DataFrame({"mes": d["sar_mensual"].index.strftime("%Y-%m"), "serie": nm, "SARIMAX": d["sar_mensual"].values.round(2)})
    if d["prophet"] is not None: base["Prophet"] = d["prophet"]["serie_mensual"].values.round(2)
    filas_pron.append(base)
reg_tabla("ts_pronostico_12_meses", pd.concat(filas_pron))
ys_an = serie_d["consumo_kwh_dia"]; stl_d = STL(ys_an, period=7, robust=True).fit()
mediana_mov = ys_an.rolling(15, center=True, min_periods=5).median()
feat_an = pd.DataFrame({"residuo_stl": stl_d.resid, "z_robusto": (ys_an - mediana_mov) / (1.4826 * (ys_an - mediana_mov).abs().median()),
                        "desv_rel_15d": (ys_an - mediana_mov) / mediana_mov, "dif_1d": ys_an.diff().fillna(0)}, index=ys_an.index)
IFO = IsolationForest(n_estimators=400, contamination=0.02, random_state=SEED).fit(feat_an)
serie_d["anomalia_detectada"] = (IFO.predict(feat_an) == -1).astype(int)
serie_d["score_anomalia"] = -IFO.score_samples(feat_an)
real = serie_d["anomalia_real"].values; det = serie_d["anomalia_detectada"].values
pr_, rc_, f1_ = precision_score(real, det, zero_division=0), recall_score(real, det, zero_division=0), f1_score(real, det, zero_division=0)
reg_tabla("ts_anomalias_detectadas", serie_d.loc[serie_d.anomalia_detectada == 1, ["consumo_kwh_dia", "anomalia_real", "score_anomalia"]].round(3), index=True)
fig_an = go.Figure()
fig_an.add_trace(go.Scatter(x=ys_an.index, y=ys_an.values, mode="lines", name="Consumo diario", line=dict(color="#7f8c8d", width=1)))
tp = serie_d[(serie_d.anomalia_detectada == 1) & (serie_d.anomalia_real == 1)]; fp = serie_d[(serie_d.anomalia_detectada == 1) & (serie_d.anomalia_real == 0)]
fn = serie_d[(serie_d.anomalia_detectada == 0) & (serie_d.anomalia_real == 1)]
fig_an.add_trace(go.Scatter(x=tp.index, y=tp.consumo_kwh_dia, mode="markers", name="Detectada y real", marker=dict(color="#1e8449", size=10, symbol="diamond")))
fig_an.add_trace(go.Scatter(x=fp.index, y=fp.consumo_kwh_dia, mode="markers", name="Falsa alarma", marker=dict(color="#e6a817", size=9, symbol="x")))
fig_an.add_trace(go.Scatter(x=fn.index, y=fn.consumo_kwh_dia, mode="markers", name="No detectada", marker=dict(color="#c0392b", size=10, symbol="circle-open", line=dict(width=2))))
fig_an.update_layout(title=f"Isolation Forest | P={pr_:.2f}, R={rc_:.2f}, F1={f1_:.2f}", height=500, yaxis_title="kWh/día")
reg_fig("ts_anomalias", fig_an, clave=True)
log("Series temporales completadas")

# =============================================================================
# BLOQUE 7. OPTIMIZACIÓN
# =============================================================================
log("BLOQUE 7: Optimización de presupuesto")
POB_MUNICIPIO, PERSONAS_HOGAR = 320_000, 3.6
HOGARES_MUN = int(POB_MUNICIPIO / PERSONAS_HOGAR)
PESO_ESTRATO = {1: 0.65, 2: 0.20, 3: 0.15}
FE_CO2_KG_KWH = 0.126
VIDA_UTIL, DEGRAD = 25, 0.005
FACTOR_VIDA = sum((1 - DEGRAD) ** t for t in range(VIDA_UTIL))
SUBSIDIO_DEF = {1: 0.80, 2: 0.50, 3: 0.20}
PAR = {}
for s in (1, 2, 3):
    d = df[df.estrato == s]; dsi = d[d.dispuesto_instalar_paneles == "Sí"]
    base = dsi if len(dsi) >= 10 else d
    p_si = float((d.dispuesto_instalar_paneles == "Sí").mean())
    PAR[s] = dict(inversion=float(base["inversion_estimada_cop"].mean()), ahorro_anual=float(base["ahorro_anual_cop"].mean()),
                  gen_anual_kwh=float(base["generacion_mensual_estimada_kwh"].mean() * 12), p_si=p_si,
                  elegibles=int(HOGARES_MUN * PESO_ESTRATO[s] * p_si))
    PAR[s]["energia_vida_kwh"] = PAR[s]["gen_anual_kwh"] * FACTOR_VIDA
    PAR[s]["co2_vida_t"] = PAR[s]["energia_vida_kwh"] * FE_CO2_KG_KWH / 1000.0
    PAR[s]["ahorro_10a_cop"] = PAR[s]["ahorro_anual"] * sum((1.05 * (1 - DEGRAD)) ** t for t in range(10))
reg_tabla("optimizacion_parametros_por_estrato", pd.DataFrame(PAR).T.round(2), index=True)
SOLVER_USADO = {"nombre": None}
def _resolver_pulp(beneficio, costo, caps, presupuesto, min_share):
    prob = pulp.LpProblem("Subsidios_FV_Popayan", pulp.LpMaximize)
    x = {s: pulp.LpVariable(f"E{s}", 0, caps[s], pulp.LpInteger) for s in (1, 2, 3)}
    prob += pulp.lpSum(beneficio[s] * x[s] for s in x), "Beneficio_total"
    prob += pulp.lpSum(costo[s] * x[s] for s in x) <= presupuesto, "Presupuesto"
    if min_share > 0:
        prob += costo[1] * x[1] - min_share * pulp.lpSum(costo[s] * x[s] for s in x) >= 0, "Equidad_E1"
    prob.solve(pulp.PULP_CBC_CMD(msg=0))
    if pulp.LpStatus[prob.status] != "Optimal":
        return None, pulp.LpStatus[prob.status]
    return {s: int(round(x[s].value() or 0)) for s in x}, "Optimal"
def _resolver_scipy(beneficio, costo, caps, presupuesto, min_share):
    from scipy.optimize import milp, LinearConstraint, Bounds
    c = -np.array([beneficio[s] for s in (1, 2, 3)])
    co = np.array([costo[s] for s in (1, 2, 3)])
    A = [co]; ub = [presupuesto]
    if min_share > 0:
        fila = min_share * co.copy(); fila[0] = -(1 - min_share) * co[0]
        A.append(fila); ub.append(0.0)
    r = milp(c, constraints=LinearConstraint(np.array(A), -np.inf, np.array(ub)), integrality=np.ones(3),
             bounds=Bounds(np.zeros(3), np.array([caps[s] for s in (1, 2, 3)], float)))
    if not r.success:
        return None, r.message
    return {s: int(round(r.x[i])) for i, s in enumerate((1, 2, 3))}, "Optimal"
def _resolver(beneficio, costo, caps, presupuesto, min_share):
    try:
        if int(str(pulp.__version__).split(".")[0]) >= 4:
            raise RuntimeError("PuLP>=4")
        sol = _resolver_pulp(beneficio, costo, caps, presupuesto, min_share); SOLVER_USADO["nombre"] = "PuLP + CBC"
        return sol
    except Exception:
        SOLVER_USADO["nombre"] = "scipy.optimize.milp (HiGHS)"
        return _resolver_scipy(beneficio, costo, caps, presupuesto, min_share)
def optimizar(presupuesto_mm=1000.0, sub1=0.80, sub2=0.50, sub3=0.20, min_share_e1=0.0, w_en=0.5, w_hog=0.3, w_ah=0.2):
    sub = {1: sub1, 2: sub2, 3: sub3}
    tw = max(w_en + w_hog + w_ah, 1e-9); w_en, w_hog, w_ah = w_en / tw, w_hog / tw, w_ah / tw
    costo = {s: PAR[s]["inversion"] * sub[s] for s in sub}
    en_max = max(PAR[s]["energia_vida_kwh"] for s in sub); ah_max = max(PAR[s]["ahorro_10a_cop"] for s in sub)
    beneficio = {s: w_en * PAR[s]["energia_vida_kwh"] / en_max + w_hog + w_ah * PAR[s]["ahorro_10a_cop"] / ah_max for s in sub}
    caps = {s: PAR[s]["elegibles"] for s in sub}
    sol, estado = _resolver(beneficio, costo, caps, presupuesto_mm * 1e6, min_share_e1)
    if sol is None:
        return None, str(estado)
    filas = []
    for s in (1, 2, 3):
        n = int(sol[s])
        filas.append({"estrato": f"E{s}", "subsidio_%": int(sub[s] * 100), "sistemas": n, "subsidio_por_sistema_cop": round(costo[s]), "gasto_municipal_cop": round(costo[s] * n),
                      "inversion_total_cop": round(PAR[s]["inversion"] * n), "hogares_atendidos": n, "energia_vida_util_MWh": round(PAR[s]["energia_vida_kwh"] * n / 1000, 1),
                      "CO2_evitado_t": round(PAR[s]["co2_vida_t"] * n, 1), "ahorro_familias_10a_cop": round(PAR[s]["ahorro_10a_cop"] * n)})
    res = pd.DataFrame(filas)
    tot = {"estrato": "TOTAL", "subsidio_%": np.nan, "sistemas": res.sistemas.sum(), "subsidio_por_sistema_cop": np.nan, "gasto_municipal_cop": res.gasto_municipal_cop.sum(),
           "inversion_total_cop": res.inversion_total_cop.sum(), "hogares_atendidos": res.hogares_atendidos.sum(), "energia_vida_util_MWh": res.energia_vida_util_MWh.sum(),
           "CO2_evitado_t": res.CO2_evitado_t.sum(), "ahorro_familias_10a_cop": res.ahorro_familias_10a_cop.sum()}
    res = pd.concat([res, pd.DataFrame([tot])], ignore_index=True)
    return res, "Optimal"
def fig_optimizacion(res, titulo):
    r = res[res.estrato != "TOTAL"]
    fig = make_subplots(rows=1, cols=3, subplot_titles=("Sistemas", "Gasto municipal (M)", "CO₂ (t)"))
    cols = [COLOR_E[1], COLOR_E[2], COLOR_E[3]]
    fig.add_bar(x=r.estrato, y=r.sistemas, marker_color=cols, showlegend=False, text=r.sistemas, row=1, col=1)
    fig.add_bar(x=r.estrato, y=(r.gasto_municipal_cop / 1e6).round(1), marker_color=cols, showlegend=False, text=(r.gasto_municipal_cop / 1e6).round(1), row=1, col=2)
    fig.add_bar(x=r.estrato, y=r.CO2_evitado_t, marker_color=cols, showlegend=False, text=r.CO2_evitado_t, row=1, col=3)
    fig.update_layout(height=430, title=titulo, title_font_size=14)
    return fig
def ejecutar_optimizacion(presupuesto_mm, sub1, sub2, sub3, min_e1, w_en, w_hog, w_ah):
    res, est = optimizar(presupuesto_mm, sub1, sub2, sub3, min_e1 / 100.0, w_en, w_hog, w_ah)
    if res is None:
        return f"⚠️ Sin solución ({est})", go.Figure(), pd.DataFrame()
    t = res.iloc[-1]
    md = (f"### Óptimo ({est})\n- Presupuesto: **${presupuesto_mm:,.0f} M** | gasto: **${t.gasto_municipal_cop/1e6:,.1f} M** | sistemas: **{int(t.sistemas):,}**\n"
          f"- Energía vida útil: **{t.energia_vida_util_MWh:,.0f} MWh** | CO₂: **{t.CO2_evitado_t:,.0f} tCO₂**\n- Ahorro 10a familias: **${t.ahorro_familias_10a_cop/1e6:,.0f} M**")
    return md, fig_optimizacion(res, f"Asignación óptima ${presupuesto_mm:,.0f} M"), res
res_base, _ = optimizar(1000, *SUBSIDIO_DEF.values(), 0.0)
res_eq, _ = optimizar(1000, *SUBSIDIO_DEF.values(), 0.50)
reg_tabla("optimizacion_sin_equidad", res_base); reg_tabla("optimizacion_con_equidad_50", res_eq)
reg_fig("opt_sin_equidad", fig_optimizacion(res_base, "Sin equidad"), clave=True)
reg_fig("opt_con_equidad", fig_optimizacion(res_eq, "Con equidad ≥50% E1"))
print("Solver:", SOLVER_USADO["nombre"])
filas_f = []
for m_ in range(0, 100, 10):
    r_, _ = optimizar(1000, *SUBSIDIO_DEF.values(), m_ / 100.0)
    if r_ is not None:
        t = r_.iloc[-1]; filas_f.append({"min_share_E1_%": m_, "hogares": t.hogares_atendidos, "CO2_t": t.CO2_evitado_t})
tab_f = pd.DataFrame(filas_f); reg_tabla("optimizacion_frontera_equidad", tab_f)
fig_front = make_subplots(specs=[[{"secondary_y": True}]])
fig_front.add_scatter(x=tab_f["min_share_E1_%"], y=tab_f["CO2_t"], mode="lines+markers", name="CO₂ evitado", secondary_y=False)
fig_front.add_scatter(x=tab_f["min_share_E1_%"], y=tab_f["hogares"], mode="lines+markers", name="Hogares", secondary_y=True)
fig_front.update_layout(title="Frontera eficiencia-equidad", height=430)
reg_fig("opt_frontera_equidad", fig_front)
filas_p = []
for pm in (250, 500, 750, 1000, 1500, 2000, 3000):
    r_, _ = optimizar(pm, *SUBSIDIO_DEF.values(), 0.30)
    if r_ is not None:
        for _, f in r_[r_.estrato != "TOTAL"].iterrows(): filas_p.append({"presupuesto_M": pm, "estrato": f.estrato, "sistemas": f.sistemas})
fig_pres = px.bar(pd.DataFrame(filas_p), x="presupuesto_M", y="sistemas", color="estrato", color_discrete_map={"E1": COLOR_E[1], "E2": COLOR_E[2], "E3": COLOR_E[3]},
                  title="Sensibilidad al presupuesto", barmode="stack")
fig_pres.update_layout(height=430); reg_fig("opt_sensibilidad_presupuesto", fig_pres)
log("Optimización completada")

# =============================================================================
# BLOQUE 8. ECONOMÍA CIRCULAR
# =============================================================================
log("BLOQUE 8: RAEE")
COMPOSICION = {"Vidrio": 0.70, "Aluminio": 0.18, "Polímeros": 0.07, "Silicio": 0.03, "Cobre": 0.01, "Otros": 0.01}
RECUPERACION = {"Vidrio": 0.85, "Aluminio": 0.95, "Polímeros": 0.0, "Silicio": 0.70, "Cobre": 0.90, "Otros": 0.50}
KG_POR_W = 0.052
MASA_INVERSOR_KG = 8.0
MODOS_RAEE = ["Weibull regular (α=5.38, β=30 años)", "Pérdida temprana (α=2.5, β=18 años)", "Retiro programado a la vida útil"]
def _F(t, a, b): return 1 - np.exp(-(np.maximum(t, 0) / b) ** a)
def simular_raee(total_sistemas=5000, potencia_w=450, paneles_por_sistema=3, modo=MODOS_RAEE[0], vida=25, factor_recuperacion=1.0, anio_ini=2026, anio_fin=2030, horizonte=2065):
    años = np.arange(anio_ini, horizonte + 1); cohortes = np.arange(anio_ini, anio_fin + 1)
    w = 1 / (1 + np.exp(-(np.linspace(-2, 2, len(cohortes))))); inst = total_sistemas * w / w.sum()
    masa_panel = potencia_w * KG_POR_W
    fin_pan = np.zeros(len(años)); fin_inv = np.zeros(len(años))
    for c, n_sis in zip(cohortes, inst):
        n_pan = n_sis * paneles_por_sistema
        for j, y in enumerate(años):
            edad = y - c
            if edad < 0: continue
            if modo.startswith("Retiro"):
                f = 1.0 if edad == vida else 0.0
            else:
                a, b = (5.38, 30.0) if modo.startswith("Weibull") else (2.5, 18.0)
                f = _F(edad + 1, a, b) - _F(edad, a, b)
            f_inv = _F(edad + 1, 3.0, 13.0) - _F(edad, 3.0, 13.0)
            fin_pan[j] += n_pan * f; fin_inv[j] += n_sis * f_inv
    res = pd.DataFrame({"año": años, "paneles_fin_vida": fin_pan, "inversores_fin_vida": fin_inv})
    res["residuo_paneles_t"] = res.paneles_fin_vida * masa_panel / 1000.0
    res["residuo_inversores_t"] = res.inversores_fin_vida * MASA_INVERSOR_KG / 1000.0
    res["residuo_total_t"] = res.residuo_paneles_t + res.residuo_inversores_t
    for mat, frac in COMPOSICION.items():
        res[f"{mat}_recuperable_t"] = res.residuo_paneles_t * frac * min(1.0, RECUPERACION[mat] * factor_recuperacion)
    res["recuperable_total_t"] = res[[c for c in res.columns if c.endswith("_recuperable_t")]].sum(axis=1)
    res["a_disposicion_final_t"] = res.residuo_paneles_t - res["recuperable_total_t"] + res.residuo_inversores_t
    return res
def fig_raee(res, titulo):
    fig = make_subplots(specs=[[{"secondary_y": True}]])
    fig.add_bar(x=res["año"], y=res["recuperable_total_t"], name="Recuperables", marker_color="#1e8449", secondary_y=False)
    fig.add_bar(x=res["año"], y=res["a_disposicion_final_t"], name="Disposición final", marker_color="#c0392b", secondary_y=False)
    fig.add_scatter(x=res["año"], y=res["residuo_total_t"].cumsum(), name="Acumulado", mode="lines", line=dict(color="black"), secondary_y=True)
    fig.update_layout(barmode="stack", title=titulo, height=500, xaxis_title="Año")
    return fig
def ejecutar_raee(total, pot, npan, modo, vida, rec_pct):
    r = simular_raee(int(total), float(pot), int(npan), modo, int(vida), rec_pct / 100.0)
    pico = r.loc[r.residuo_total_t.idxmax()]
    md = (f"### RAEE 2026-2065\n- Sistemas: **{int(total):,}** | RAEE total: **{r.residuo_total_t.sum():,.1f} t**\n- Año pico: **{int(pico['año'])}** con {pico.residuo_total_t:,.1f} t\n"
          f"- Recuperables: **{r.recuperable_total_t.sum():,.1f} t** ({100*r.recuperable_total_t.sum()/max(r.residuo_total_t.sum(),1e-9):.0f}%)")
    tab = r[r.residuo_total_t > 0.0005][["año", "residuo_total_t", "recuperable_total_t", "a_disposicion_final_t"]].round(3)
    if tab.empty: tab = r[["año", "residuo_total_t"]].round(3)
    return md, fig_raee(r, f"RAEE - {modo}"), tab
for i, m in enumerate(MODOS_RAEE):
    r_ = simular_raee(5000, 450, 3, m)
    reg_fig(f"raee_escenario_{i+1}", fig_raee(r_, f"RAEE - {m}"), clave=(i == 2)); reg_tabla(f"raee_escenario_{i+1}", r_.round(4))
kwp_t = float(df["generacion_potencial_kwp"].mean()); gen_anual = float(df["generacion_mensual_estimada_kwh"].mean() * 12)
años_v = np.arange(0, VIDA_UTIL + 1); gen_acum = np.concatenate([[0], np.cumsum(gen_anual * (1 - DEGRAD) ** np.arange(VIDA_UTIL))])
fig_vida = make_subplots(specs=[[{"secondary_y": True}]])
fig_vida.add_scatter(x=años_v, y=gen_acum / 1000, name="Generación acum. (MWh)", mode="lines", secondary_y=False)
fig_vida.add_scatter(x=años_v, y=gen_acum * FE_CO2_KG_KWH / 1000, name="CO₂ evitado (t)", mode="lines", line=dict(dash="dash"), secondary_y=True)
fig_vida.update_layout(title=f"Ciclo de vida ({kwp_t:.2f} kWp, degradación {DEGRAD*100:.1f}%/año)", height=420, xaxis_title="Año")
reg_fig("raee_ciclo_vida", fig_vida)
log("RAEE completado")

# =============================================================================
# SIMULADORES
# =============================================================================
def _mediana(c): return float(df[c].median())
def _moda(c): return df[c].mode().iloc[0]
OPC = {c: sorted(df[c].dropna().unique().tolist()) for c in CAT_CLS + ["orientacion_cubierta"]}

def predecir_adopcion(estrato, zona, personas, menores, mayores, ingreso, consumo, tarifa, area_viv, area_cub, n_pan, p_pan, pct_ahorro, retorno,
                      educ, ocup, empresa, tcub, sombra, internet, smart, medidor, subsidio, reciclaje, comunidad):
    if menores + mayores >= personas:
        return {"Error": 1.0}, go.Figure(), "⚠️ Menores + adultos mayores debe ser menor que personas."

    # PERFIL HORARIO AUTOMÁTICO
    perfil_base = np.array([
        0.018, 0.015, 0.014, 0.014, 0.015, 0.020,   # 0-5 madrugada
        0.035, 0.055, 0.060, 0.045, 0.038, 0.040,   # 6-11 mañana
        0.045, 0.040, 0.038, 0.038, 0.042, 0.048,   # 12-17 tarde
        0.060, 0.070, 0.065, 0.055, 0.040, 0.025    # 18-23 noche
    ])
    perfil_base = perfil_base / perfil_base.sum()
    consumo_diario = consumo / 30.0
    kWh_horas = (perfil_base * consumo_diario)

    fila_dict = {"estrato": int(estrato), "zona": zona, "num_personas": personas, "num_menores": menores, "num_adultos_mayores": mayores,
                 "ingreso_mensual_cop": ingreso, "consumo_kwh_mes": consumo, "tarifa_kwh_cop": tarifa, "area_vivienda_m2": area_viv, "area_cubierta_m2": area_cub,
                 "num_paneles_estimados": n_pan, "potencia_panel_w": p_pan, "porcentaje_ahorro": pct_ahorro, "retorno_simple_anios": retorno,
                 "nivel_educativo": educ, "ocupacion_principal": ocup, "tiene_empresa": empresa, "tipo_cubierta": tcub, "sombras_presentes": sombra,
                 "tiene_internet": internet, "tiene_smartphone": smart, "tiene_medidor_inteligente": medidor, "recibe_subsidio": subsidio,
                 "conoce_reciclaje_paneles": reciclaje, "interes_comunidad_energetica": comunidad}

    for h, val in zip(HORAS, kWh_horas):
        fila_dict[h] = float(val)

    fila = pd.DataFrame([fila_dict])
    fila = agregar_perfil(ingenieria_caracteristicas(fila))
    fila_full = fila[FEATS_CLS_FULL].copy()
    p = MODELO_FINAL_CLF.predict_proba(fila_full)[0]

    dic = {CLASES[i]: float(p[i]) for i in range(3)}
    fig = go.Figure(go.Bar(x=CLASES, y=[dic[c] * 100 for c in CLASES], marker_color=[COLOR_T[c] for c in CLASES],
                            text=[f"{dic[c]*100:.1f}%" for c in CLASES], textposition="auto"))
    fig.update_layout(title="Probabilidades de adopción", yaxis_title="%", yaxis_range=[0, 100], height=380)
    mejor = max(dic, key=dic.get)
    consejo = {"Sí": "Hogar prioritario.", "Tal vez": "Hogar indeciso: simulación y capacitación.", "No": "Priorizar subsidios."}[mejor]

    return dic, fig, f"**{mejor}** ({dic[mejor]*100:.1f}%). {consejo}\n\n*Modelo: {MEJOR}{' + calibración' if USAR_CAL else ''}.*"

def predecir_ahorro(estrato, zona, personas, consumo, tarifa, area_cub, inclin, orient, tcub, sombra, n_pan, p_pan, ratio_dia):
    kwp = n_pan * p_pan / 1000.0; area_req = n_pan * p_pan / 205.0
    aviso = "" if area_req <= 0.75 * area_cub else f"\n\n⚠️ Área requerida {area_req:.1f} m² > 75% de cubierta."
    fila = pd.DataFrame([{"estrato": int(estrato), "num_personas": personas, "consumo_kwh_mes": consumo, "tarifa_kwh_cop": tarifa, "area_cubierta_m2": area_cub,
                          "inclinacion_cubierta_grados": inclin, "num_paneles_estimados": n_pan, "potencia_panel_w": p_pan, "ratio_diurno_6_17": ratio_dia / 100.0,
                          "zona": zona, "tipo_cubierta": tcub, "orientacion_cubierta": orient, "sombras_presentes": sombra}])[FEATS_REG]
    ahorro = float(REG["ahorro_anual_cop"]["modelo"].predict(fila)[0]); ret = float(REG["retorno_simple_anios"]["modelo"].predict(fila)[0])
    factura_anual = consumo * tarifa * 12; ahorro = float(np.clip(ahorro, 0, 0.8 * factura_anual)); inv = kwp * COSTO_KWP_REF
    fig = go.Figure(go.Bar(x=["Factura anual", "Ahorro", "Con FV"], y=[factura_anual, ahorro, factura_anual - ahorro],
                           marker_color=["#7f8c8d", "#1e8449", "#2471a3"], text=[f"${v/1e6:.2f}M" for v in (factura_anual, ahorro, factura_anual - ahorro)], textposition="auto"))
    fig.update_layout(title="Impacto económico anual (COP)", height=380)
    md = (f"### Resultados\n- {kwp:.2f} kWp ({n_pan}×{p_pan} W) | área {area_req:.1f} m²\n- Ahorro anual: **${ahorro:,.0f}** ({100*ahorro/factura_anual:.0f}%)\n"
          f"- Retorno: **{ret:.1f} años** | inversión ~ **${inv:,.0f}**{aviso}")
    return md, fig

# =============================================================================
# BLOQUE 10. EXPORTACIÓN
# =============================================================================
log("BLOQUE 10: Exportación")
def guardar_modelo(obj, nombre):
    ruta = f"{DIRS['modelos']}/{nombre}.joblib"
    try:
        joblib.dump(obj, ruta, compress=3); MODELOS[nombre] = ruta
    except Exception as e:
        print("  (no se pudo guardar", nombre, ":", str(e)[:80], ")")
for n, p in PIPES.items(): guardar_modelo(p, "clf_" + re.sub(r"[^A-Za-z0-9]+", "_", n).strip("_"))
guardar_modelo(MODELO_FINAL_CLF, "clf_modelo_final_simulador")
for tgt in TARGETS_REG: guardar_modelo(REG[tgt]["modelo"], f"reg_{tgt}")
guardar_modelo({"kmeans": KM, "scaler": sc_cl, "pca2d": pca_cl, "variables": CL_FEATS, "nombres_segmentos": NOMBRES_CL}, "clustering_kmeans")
guardar_modelo({"isolation_forest": IFO, "variables": list(feat_an.columns)}, "anomalias_isolation_forest")
for nm, d in TS.items():
    guardar_modelo(d["sarimax"], "ts_sarimax_" + ("consumo" if nm == "Consumo" else "generacion"))
    try: d["sarimax"].remove_data()
    except Exception: pass
    if d["prophet"] is not None:
        try:
            from prophet.serialize import model_to_json
            ruta = f"{DIRS['modelos']}/ts_prophet_{'consumo' if nm == 'Consumo' else 'generacion'}.json"
            with open(ruta, "w") as fh: fh.write(model_to_json(d["prophet"]["modelo"]))
            MODELOS["ts_prophet_" + nm] = ruta
        except Exception as e:
            print("  Prophet no serializado:", str(e)[:60])
if NN_OK and isinstance(PIPES.get("Red Neuronal Multimodal (PyTorch)"), TorchTabularClassifier):
    ruta_pt = f"{DIRS['modelos']}/nn_hogar_adopcion.pt"
    try:
        torch.save(PIPES["Red Neuronal Multimodal (PyTorch)"].model_.state_dict(), ruta_pt)
        MODELOS["nn_hogar_adopcion_pt"] = ruta_pt
    except Exception as e:
        print("  state_dict no guardado:", str(e)[:80])
import sklearn, plotly, statsmodels
meta = {"semilla": SEED, "clases": CLASES, "features_clasificacion": FEATS_CLS, "hora_features": HORA_FEATS,
        "features_clasificacion_full": FEATS_CLS_FULL, "features_regresion": FEATS_REG,
        "mejor_clasificador": MEJOR, "calibrado": bool(USAR_CAL),
        "mejores_regresores": {t: REG[t]["mejor"] for t in REG}, "k_kmeans": K_OPT,
        "arquitectura_nn": {"nombre": "HogarAdopcionNet", "tipo": "multimodal (MLP + embeddings + Conv1d)",
                             "disponible": bool(NN_OK), "device": str(DEVICE) if NN_OK else "cpu"},
        "versiones": {"sklearn": sklearn.__version__, "pandas": pd.__version__, "numpy": np.__version__,
                      "plotly": plotly.__version__, "statsmodels": statsmodels.__version__, "gradio": gr.__version__,
                      **({"torch": torch.__version__} if NN_OK else {})},
        "aviso": "Datos sintéticos. Cumplir Ley 1581/2012 con datos reales."}
with open(f"{OUT}/metadatos.json", "w", encoding="utf-8") as fh: json.dump(meta, fh, ensure_ascii=False, indent=2)
for f in glob.glob(f"{OUT}/*.zip"): os.remove(f)
ZIP_PATH = f"{OUT}/resultados_proyecto_solar_popayan.zip"
with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as z:
    for carpeta, _, archivos in os.walk(OUT):
        for a in archivos:
            ruta = os.path.join(carpeta, a)
            if ruta != ZIP_PATH: z.write(ruta, os.path.relpath(ruta, OUT))
print(f"ZIP: {ZIP_PATH} ({os.path.getsize(ZIP_PATH)/1e6:.1f} MB) | {len(TABLAS)} tablas | {len(FIGS)} gráficos | {len(MODELOS)} modelos")

# =============================================================================
# BLOQUE 9. DASHBOARD MINIMALISTA
# =============================================================================
log("BLOQUE 9: Dashboard Gradio")
def _fmt_df(d, nd=3):
    d = d.copy()
    for c in d.columns:
        if pd.api.types.is_float_dtype(d[c]): d[c] = d[c].round(nd)
    return d
m_b = tab_te.loc[MEJOR]
n_si = int((df.dispuesto_instalar_paneles == "Sí").sum())
TXT_PORTADA = f"""
# ☀️ Solar Popayán 2026-2030 · IA con Red Neuronal Multimodal
> ⚠️ **Datos sintéticos.** {len(df):,} hogares. Con datos reales aplicaría Ley 1581/2012.

**Modelos:** 6 clásicos + **Red Neuronal Multimodal PyTorch** (MLP numérico + embeddings categóricos + Conv1d 24h).
**Mejor (F1-macro CV):** {MEJOR} · AUC macro {m_b.roc_auc_ovr:.3f} · F1-macro test {m_b.f1_macro:.3f}.

**Cifras:** {n_si} hogares "Sí" ({100*n_si/len(df):.0f}%) · consumo medio {df.consumo_kwh_mes.mean():.0f} kWh/mes · retorno medio {df.retorno_simple_anios.mean():.1f} años.
"""
def fig_corr_metodo(metodo):
    c = df[NUM_EDA].corr(method=metodo.lower()).round(2)
    f = px.imshow(c, zmin=-1, zmax=1, color_continuous_scale="RdBu_r", aspect="auto", title=f"Correlación ({metodo})"); f.update_layout(height=780)
    return f
def vista_multivariada(op):
    return {"PCA 2D": fig_pca2, "PCA 3D": fig_pca3, "t-SNE": fig_tsne, "Varianza (PCA)": fig_var, "Dendrograma": fig_dend0}[op]
def pareto_ui(poblacion, nivel):
    f, t = fig_pareto(poblacion, "Barrera" if nivel.startswith("Barrera") else "Segmentos")
    return f, t
def fairness_ui(col):
    t, r = FAIR[col]; f = fig_fairness(col)
    md = "**Disparidades:**\n\n" + "\n".join(f"- {k}: **{v}**" for k, v in r.items())
    return f, _fmt_df(t.reset_index()), md
def shap_force_ui(clase, idx): return fig_shap_force(int(idx), clase)
def shap_bee_ui(clase): return fig_shap_beeswarm(clase)
def reg_diag_ui(tgt_label):
    tgt = [k for k, v in TARGETS_REG.items() if v == tgt_label][0]
    return FIGS[f"reg_diagnostico_{tgt}"], _fmt_df(REG[tgt]["tabla"].reset_index(), 4)
def stl_ui(serie, frec): return fig_stl(serie, frec), fig_acf_pacf(serie, frec)
def pron_ui(serie): return fig_pronostico(serie)
def uni_ui(v): return fig_univariado(v)
def biv_ui(x, y, c): return fig_bivariado(x, y, c)
def slider(lbl, col, lo, hi, step=1, escala=1.0):
    return gr.Slider(minimum=lo, maximum=hi, value=float(np.clip(df[col].median() / escala, lo, hi)), step=step, label=lbl)

# Tema minimalista sin .set() para evitar errores de compatibilidad
tema_minimalista = gr.themes.Soft(
    primary_hue="orange",
    secondary_hue="blue",
    neutral_hue="slate",
    font=[gr.themes.GoogleFont("Inter"), "ui-sans-serif", "system-ui", "sans-serif"]
)

with gr.Blocks(title="Solar Popayán 2026-2030 | NN multimodal", theme=tema_minimalista, css=".gradio-container {max-width: 1200px !important} .block {box-shadow: none !important;}") as demo:
    gr.Markdown("# ☀️ Solar Popayán 2026-2030 · IA con Red Neuronal Multimodal")
    with gr.Tabs():
        with gr.TabItem("1. Portada"):
            gr.Markdown(TXT_PORTADA)
        with gr.TabItem("2. Diccionario"):
            gr.Dataframe(value=diccionario, interactive=False, wrap=True)
            gr.Markdown("#### Coherencia")
            gr.Dataframe(value=calidad, interactive=False)
        with gr.TabItem("3. EDA Univariado"):
            d3 = gr.Dropdown(choices=NUM_EDA + CAT_EDA, value="consumo_kwh_mes", label="Variable")
            p3 = gr.Plot(value=fig_univariado("consumo_kwh_mes")); d3.change(uni_ui, d3, p3)
            gr.Dataframe(value=_fmt_df(desc_num.reset_index().rename(columns={"index": "variable"})), interactive=False)
        with gr.TabItem("4. EDA Bivariado"):
            with gr.Row():
                x4 = gr.Dropdown(choices=NUM_EDA + CAT_EDA, value="ingreso_mensual_cop", label="X")
                y4 = gr.Dropdown(choices=NUM_EDA + CAT_EDA, value="consumo_kwh_mes", label="Y")
                c4 = gr.Dropdown(choices=["(ninguno)"] + CAT_EDA, value="estrato", label="Color")
            b4 = gr.Button("Graficar", variant="primary"); p4 = gr.Plot(value=FIGS["eda_bivariado_consumo_ingreso"])
            b4.click(biv_ui, [x4, y4, c4], p4)
        with gr.TabItem("5. EDA Multivariado"):
            r5 = gr.Radio(["PCA 2D", "PCA 3D", "t-SNE", "Varianza (PCA)", "Dendrograma"], value="PCA 2D", label="Vista")
            p5 = gr.Plot(value=fig_pca2); r5.change(vista_multivariada, r5, p5)
            gr.Dataframe(value=cargas.reset_index().rename(columns={"index": "variable"}), interactive=False)
        with gr.TabItem("6. Correlación"):
            r6 = gr.Radio(["Spearman", "Pearson"], value="Spearman", label="Método"); p6 = gr.Plot(value=fig_corr); r6.change(fig_corr_metodo, r6, p6)
        with gr.TabItem("7. Target"):
            gr.Plot(value=FIGS["analisis_target"])
        with gr.TabItem("8. Pareto"):
            with gr.Row():
                r8a = gr.Radio(["No + Tal vez", "Solo No", "Todos"], value="No + Tal vez", label="Población")
                r8b = gr.Radio(["Barrera", "Segmentos"], value="Barrera", label="Nivel")
            p8 = gr.Plot(value=fig_par); t8 = gr.Dataframe(value=tabla_par, interactive=False)
            r8a.change(pareto_ui, [r8a, r8b], [p8, t8]); r8b.change(pareto_ui, [r8a, r8b], [p8, t8])
        with gr.TabItem("9. Comparación Modelos"):
            gr.Plot(value=fig_cmp); gr.Plot(value=fig_cv)
            cols9 = ["cv_f1_macro", "cv_roc_auc_ovr", "accuracy", "f1_macro", "roc_auc_ovr", "pr_auc_macro", "mcc", "brier"]
            gr.Dataframe(value=_fmt_df(tab_clf[cols9].reset_index()), interactive=False)
        if NN_OK:
            with gr.TabItem("9b. Red Neuronal"):
                gr.Markdown("Arquitectura **HogarAdopcionNet**: MLP numérico + embeddings categóricos + Conv1d(24h) → fusión → 3 clases.")
                if "nn_curvas_entrenamiento" in FIGS: gr.Plot(value=FIGS["nn_curvas_entrenamiento"])
                gr.Markdown(f"Device: **{DEVICE}**")
        with gr.TabItem("10. Métricas"):
            gr.Markdown(f"### {MEJOR}\n" + " · ".join(f"**{k}**={m_b[k]:.3f}" for k in ["accuracy", "f1_macro", "roc_auc_ovr", "pr_auc_macro", "mcc", "brier"]))
            gr.Plot(value=FIGS["clf_matriz_confusion"]); gr.Plot(value=FIGS["clf_roc_pr"])
        with gr.TabItem("11. Calibración"):
            gr.Plot(value=FIGS["clf_calibracion"]); gr.Dataframe(value=tab_cal.reset_index().rename(columns={"index": "métrica"}), interactive=False)
        with gr.TabItem("12. SHAP"):
            gr.Plot(value=FIGS["clf_shap_resumen"])
            with gr.Row():
                c12 = gr.Dropdown(CLASES, value="Sí", label="Clase")
                i12 = gr.Slider(0, len(X_te) - 1, value=0, step=1, label="Hogar")
            pb = gr.Plot(value=FIGS["clf_shap_beeswarm_si"]); pf = gr.Plot(value=FIGS["clf_shap_force_ejemplo"])
            c12.change(shap_bee_ui, c12, pb); c12.change(shap_force_ui, [c12, i12], pf); i12.change(shap_force_ui, [c12, i12], pf)
            gr.Plot(value=FIGS["clf_permutation_importance"])
        with gr.TabItem("13. Fairness"):
            d13 = gr.Dropdown(["estrato", "zona", "estrato_zona", "jefe_hogar_genero"], value="estrato", label="Grupo")
            f0, t0, m0 = fairness_ui("estrato")
            p13 = gr.Plot(value=f0); t13 = gr.Dataframe(value=t0, interactive=False); m13 = gr.Markdown(m0)
            d13.change(fairness_ui, d13, [p13, t13, m13])
        with gr.TabItem("14. Predicción Adopción"):
            gr.Markdown("### 🏠 Simulador de Adopción\nIngrese los datos del hogar. Puede expandir cada sección para ajustar los detalles.")
            with gr.Accordion("👤 1. Datos Básicos y Demográficos", open=True):
                with gr.Row():
                    s_est = gr.Dropdown([1, 2, 3], value=1, label="Estrato")
                    s_zon = gr.Dropdown(OPC["zona"], value=_moda("zona"), label="Zona")
                    s_per = slider("Personas en el hogar", "num_personas", 1, 8)
                    s_men = slider("Menores de edad", "num_menores", 0, 4)
                    s_may = slider("Adultos mayores", "num_adultos_mayores", 0, 2)
                with gr.Row():
                    s_edu = gr.Dropdown(OPC["nivel_educativo"], value=_moda("nivel_educativo"), label="Nivel Educativo")
                    s_ocu = gr.Dropdown(OPC["ocupacion_principal"], value=_moda("ocupacion_principal"), label="Ocupación")
                    s_emp = gr.Dropdown(OPC["tiene_empresa"], value="No", label="Tiene emprendimiento")
            with gr.Accordion("⚡ 2. Consumo y Viabilidad Económica", open=False):
                with gr.Row():
                    s_ing = gr.Number(value=_mediana("ingreso_mensual_cop"), label="Ingreso mensual (COP)")
                    s_con = gr.Number(value=_mediana("consumo_kwh_mes"), label="Consumo eléctrico (kWh/mes)")
                    s_tar = gr.Number(value=_mediana("tarifa_kwh_cop"), label="Tarifa (COP/kWh)")
                with gr.Row():
                    s_pa = slider("% de ahorro esperado", "porcentaje_ahorro", 30, 80, 0.5)
                    s_re = slider("Retorno simple (años)", "retorno_simple_anios", 3, 20, 0.1)
            with gr.Accordion("☀️ 3. Infraestructura y Sistema FV", open=False):
                with gr.Row():
                    s_av = gr.Number(value=_mediana("area_vivienda_m2"), label="Área vivienda (m²)")
                    s_ac = gr.Number(value=_mediana("area_cubierta_m2"), label="Área cubierta (m²)")
                    s_tc = gr.Dropdown(OPC["tipo_cubierta"], value=_moda("tipo_cubierta"), label="Material de cubierta")
                    s_som = gr.Dropdown(OPC["sombras_presentes"], value="No", label="¿Tiene sombras?")
                with gr.Row():
                    s_np = slider("Nº de paneles", "num_paneles_estimados", 3, 12)
                    s_pp = gr.Dropdown([400, 430, 450, 500, 525, 550], value=450, label="Potencia del panel (W)")
            with gr.Accordion("📱 4. Tecnología y Comunidad", open=False):
                with gr.Row():
                    s_int = gr.Dropdown(OPC["tiene_internet"], value="Sí", label="Tiene Internet")
                    s_sma = gr.Dropdown(OPC["tiene_smartphone"], value="Sí", label="Tiene Smartphone")
                    s_med = gr.Dropdown(OPC["tiene_medidor_inteligente"], value="No", label="Medidor inteligente")
                with gr.Row():
                    s_sub = gr.Dropdown(OPC["recibe_subsidio"], value="Sí", label="Recibe subsidio")
                    s_rec = gr.Dropdown(OPC["conoce_reciclaje_paneles"], value="No", label="Conoce reciclaje de paneles")
                    s_com = gr.Dropdown(OPC["interes_comunidad_energetica"], value="No", label="Interés en comunidad energética")
            with gr.Row():
                b14 = gr.Button("🚀 Predecir Adopción", variant="primary", size="lg")
            with gr.Row():
                with gr.Column():
                    l14 = gr.Label(label="Probabilidades de Adopción", num_top_classes=3)
                with gr.Column():
                    p14 = gr.Plot()
            m14 = gr.Markdown()
            ent14 = [s_est, s_zon, s_per, s_men, s_may, s_ing, s_con, s_tar, s_av, s_ac, s_np, s_pp, s_pa, s_re,
                     s_edu, s_ocu, s_emp, s_tc, s_som, s_int, s_sma, s_med, s_sub, s_rec, s_com]
            b14.click(predecir_adopcion, ent14, [l14, p14, m14])
        with gr.TabItem("15. Predicción Ahorro"):
            gr.Markdown("### 💰 Simulador Económico\nEstime el impacto financiero de la instalación fotovoltaica.")
            with gr.Accordion("Configuración del Hogar", open=True):
                with gr.Row():
                    a_est = gr.Dropdown([1, 2, 3], value=1, label="Estrato")
                    a_zon = gr.Dropdown(OPC["zona"], value=_moda("zona"), label="Zona")
                    a_per = slider("Personas", "num_personas", 1, 8)
                    a_con = gr.Number(value=_mediana("consumo_kwh_mes"), label="Consumo (kWh/mes)")
                    a_tar = gr.Number(value=_mediana("tarifa_kwh_cop"), label="Tarifa (COP/kWh)")
            with gr.Accordion("Parámetros del Sistema Solar", open=False):
                with gr.Row():
                    a_ac = gr.Number(value=_mediana("area_cubierta_m2"), label="Área cubierta (m²)")
                    a_inc = slider("Inclinación de cubierta (°)", "inclinacion_cubierta_grados", 5, 35)
                    a_ori = gr.Dropdown(OPC["orientacion_cubierta"], value=_moda("orientacion_cubierta"), label="Orientación")
                    a_tc = gr.Dropdown(OPC["tipo_cubierta"], value=_moda("tipo_cubierta"), label="Material de cubierta")
                    a_som = gr.Dropdown(OPC["sombras_presentes"], value="No", label="Sombras")
                with gr.Row():
                    a_np = slider("Nº paneles", "num_paneles_estimados", 3, 12)
                    a_pp = gr.Dropdown([400, 430, 450, 500, 525, 550], value=450, label="Potencia del panel (W)")
                    a_rd = gr.Slider(35, 70, value=round(100 * _mediana("ratio_diurno_6_17"), 1), step=0.5, label="% consumo diurno (06-18h)")
            with gr.Row():
                b15 = gr.Button("💵 Estimar Ahorro", variant="primary", size="lg")
            with gr.Row():
                with gr.Column():
                    m15 = gr.Markdown()
                with gr.Column():
                    p15 = gr.Plot()
            b15.click(predecir_ahorro, [a_est, a_zon, a_per, a_con, a_tar, a_ac, a_inc, a_ori, a_tc, a_som, a_np, a_pp, a_rd], [m15, p15])
            gr.Markdown("---")
            with gr.Accordion("📊 Ver desempeño del modelo de regresión", open=False):
                r15 = gr.Dropdown(list(TARGETS_REG.values()), value=TARGETS_REG["ahorro_anual_cop"], label="Objetivo a evaluar")
                p15b = gr.Plot(value=FIGS["reg_diagnostico_ahorro_anual_cop"])
                t15 = gr.Dataframe(value=_fmt_df(REG["ahorro_anual_cop"]["tabla"].reset_index(), 4), interactive=False)
                r15.change(reg_diag_ui, r15, [p15b, t15])
        with gr.TabItem("16. K-Means"):
            gr.Plot(value=FIGS["kmeans_seleccion_k"]); gr.Plot(value=FIGS["kmeans_pca"])
            gr.Dataframe(value=_fmt_df(perfil.reset_index(), 2), interactive=False)
            gr.Plot(value=FIGS["kmeans_perfil_heatmap"]); gr.Plot(value=FIGS["kmeans_perfil_horario"])
        with gr.TabItem("17. Jerárquico"):
            gr.Plot(value=FIGS["dendrograma_clustering"])
            gr.Dataframe(value=pd.crosstab(df["segmento"], df["cluster_jerarquico"]).reset_index(), interactive=False)
        with gr.TabItem("18. Anomalías"):
            gr.Plot(value=FIGS["dbscan_atipicos"]); gr.Dataframe(value=_fmt_df(atip), interactive=False)
            gr.Plot(value=FIGS["ts_anomalias"])
        with gr.TabItem("19. Series"):
            with gr.Row():
                d19a = gr.Dropdown(list(SERIES), value="Consumo", label="Serie"); d19b = gr.Radio(["Diaria", "Mensual"], value="Diaria", label="Frecuencia")
            gr.Plot(value=FIGS["ts_series"]); p19a = gr.Plot(value=FIGS["ts_stl_consumo_diaria"]); p19b = gr.Plot(value=FIGS["ts_acf_pacf_consumo"])
            d19a.change(stl_ui, [d19a, d19b], [p19a, p19b]); d19b.change(stl_ui, [d19a, d19b], [p19a, p19b])
        with gr.TabItem("20. Pronóstico"):
            d20 = gr.Dropdown(list(SERIES), value="Consumo", label="Serie"); p20 = gr.Plot(value=FIGS["ts_pronostico_consumo"])
            d20.change(pron_ui, d20, p20)
            gr.Dataframe(value=tab_bt, interactive=False)
            gr.Dataframe(value=TABLAS["ts_pronostico_12_meses"], interactive=False)
        with gr.TabItem("21. Optimización"):
            gr.Markdown(f"Solver: {SOLVER_USADO['nombre']}")
            with gr.Row():
                o_pre = gr.Slider(100, 5000, value=1000, step=50, label="Presupuesto (M)"); o_eq = gr.Slider(0, 90, value=0, step=5, label="Min % E1")
            with gr.Row():
                o_s1 = gr.Slider(0, 1, value=0.80, step=0.05, label="Sub E1"); o_s2 = gr.Slider(0, 1, value=0.50, step=0.05, label="Sub E2"); o_s3 = gr.Slider(0, 1, value=0.20, step=0.05, label="Sub E3")
            with gr.Row():
                o_we = gr.Slider(0, 1, value=0.5, step=0.05, label="Peso energía"); o_wh = gr.Slider(0, 1, value=0.3, step=0.05, label="Peso hogares"); o_wa = gr.Slider(0, 1, value=0.2, step=0.05, label="Peso ahorro")
            b21 = gr.Button("Optimizar", variant="primary"); m21 = gr.Markdown(); p21 = gr.Plot(); t21 = gr.Dataframe(interactive=False)
            ent21 = [o_pre, o_s1, o_s2, o_s3, o_eq, o_we, o_wh, o_wa]
            b21.click(ejecutar_optimizacion, ent21, [m21, p21, t21]); demo.load(ejecutar_optimizacion, ent21, [m21, p21, t21])
            gr.Plot(value=FIGS["opt_sin_equidad"]); gr.Plot(value=FIGS["opt_con_equidad"])
            gr.Plot(value=FIGS["opt_frontera_equidad"]); gr.Plot(value=FIGS["opt_sensibilidad_presupuesto"])
        with gr.TabItem("22. RAEE"):
            with gr.Row():
                e_tot = gr.Slider(500, 50000, value=5000, step=500, label="Sistemas"); e_pot = gr.Dropdown([400, 430, 450, 500, 525, 550], value=450, label="Potencia (W)")
                e_npa = gr.Slider(3, 12, value=3, step=1, label="Paneles/sistema")
            with gr.Row():
                e_mod = gr.Radio(MODOS_RAEE, value=MODOS_RAEE[0], label="Modelo"); e_vid = gr.Slider(20, 30, value=25, step=1, label="Vida útil")
                e_rec = gr.Slider(50, 120, value=100, step=5, label="Recuperación")
            b22 = gr.Button("Simular", variant="primary"); m22 = gr.Markdown(); p22 = gr.Plot(); t22 = gr.Dataframe(interactive=False)
            ent22 = [e_tot, e_pot, e_npa, e_mod, e_vid, e_rec]
            b22.click(ejecutar_raee, ent22, [m22, p22, t22]); demo.load(ejecutar_raee, ent22, [m22, p22, t22])
            gr.Plot(value=FIGS["raee_ciclo_vida"])
        with gr.TabItem("23. Conclusiones"):
            gr.Markdown(f"""
### Conclusiones (datos sintéticos)
1. Mejor clasificador por CV: **{MEJOR}** (AUC macro {m_b.roc_auc_ovr:.3f}, F1-macro test {m_b.f1_macro:.3f}).
2. Comparativa incluye **Red Neuronal Multimodal PyTorch** con rama Conv1d sobre perfil horario.
3. SHAP sobre **{ARBOL}**; Permutation Importance sobre el modelo final.
4. Fairness auditada sobre estrato/zona/género.

### Descargas
""")
            archivos = [ZIP_PATH] + sorted(glob.glob(f"{DIRS['tablas']}/*.csv")) + sorted(glob.glob(f"{DIRS['modelos']}/*"))
            gr.File(value=archivos, label="ZIP + CSVs + modelos", file_count="multiple", interactive=False)

print(f"Dashboard con {sum(1 for b in demo.blocks.values() if isinstance(b, gr.Tab))} pestañas.")

# Verificaciones finales
assert "Red Neuronal Multimodal (PyTorch)" in tab_clf.index, "NN no entró en tab_clf"
assert len(TABLAS) > 15, "Muy pocas tablas exportadas"
assert len(FIGS) > 25, "Muy pocos gráficos exportados"
print("✔ Verificaciones finales OK.")

print("=" * 100)
print(f"PIPELINE COMPLETO en {(time.time() - T0) / 60:.1f} min | Resultados en: {os.path.abspath(OUT)}")
print(f"Red neuronal disponible: {NN_OK} | Device: {DEVICE}")
print("ZIP:", os.path.abspath(ZIP_PATH))
print("=" * 100)
if LAUNCH:
    try:
        demo.queue().launch(share=True, debug=False, show_error=True)
    except Exception as _e:
        print("No se pudo abrir enlace público:", str(_e)[:80])
        demo.queue().launch(share=False, debug=False, show_error=True)

Instalando paquetes faltantes: ['pulp<4', 'kaleido']
[NN] PyTorch 2.14.1+cpu disponible. Device = cpu
AVISO ÉTICO: dataset SINTÉTICO. Con datos reales aplicaría Ley 1581 de 2012 y Decreto 1377 de 2013.
[   38.3s] BLOQUE 1: ETL y Feature Engineering
Dataset cargado: ./dataset_solar_popayan.csv -> 1000 filas x 99 columnas
                       regla_de_coherencia  filas_que_incumplen cumplimiento
              Sin valores nulos (por fila)                    0       100.0%
                            id_hogar único                    0       100.0%
                      Sin filas duplicadas                    0       100.0%
             Suma h00-h23 = consumo diario                    0       100.0%
                Factura = consumo x tarifa                    0       100.0%
                 Ahorro mensual <= factura                    0       100.0%
           Área requerida <= área cubierta                    0       100.0%
              Consumo anual = 12 x mensual                    

In [ ]:
# =============================================================================
# DESPLIEGUE MLOPS - EXPOSICIÓN DE GRADIO CON NGROK EN GOOGLE COLAB
# =============================================================================
import os
import sys
import time
import socket
import subprocess

# 1. Instalar pyngrok de forma silenciosa
try:
    import pyngrok
    from pyngrok import ngrok, conf
    print("✔ pyngrok ya está instalado.")
except ImportError:
    print("⚙ Instalando pyngrok...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyngrok"], check=True)
    from pyngrok import ngrok, conf
    print("✔ pyngrok instalado correctamente.")

# 2. Token de ngrok (NUNCA pegar el token en el código)
# En Colab: crea el secreto NGROK_AUTHTOKEN. En local: variable de entorno NGROK_AUTHTOKEN.
try:
    from google.colab import userdata  # type: ignore
    TOKEN_NGROK = userdata.get("NGROK_AUTHTOKEN")
except Exception:
    TOKEN_NGROK = os.environ.get("NGROK_AUTHTOKEN")

if not TOKEN_NGROK:
    print("⚠ No se encontró NGROK_AUTHTOKEN (Colab Secrets o variable de entorno).")
else:
    ngrok.set_auth_token(TOKEN_NGROK)
    print("✔ Token de ngrok configurado.")

# 3. Lanzar la aplicación Gradio (app.py) en segundo plano
PORT = 7860
APP_FILE = "app.py"

# Como tu notebook anterior es masivo y ya ejecuta una interfaz Gradio directamente,
# en caso de que no tengas un archivo fìsico 'app.py' escrito en el disco, lo creamos de prueba.
if not os.path.exists(APP_FILE):
    print(f"⚙ {APP_FILE} no encontrado en el directorio. Creando un script demo de Gradio...")
    with open(APP_FILE, "w", encoding="utf-8") as f:
        f.write("""
import gradio as gr
def demo_predict(text):
    return f"Procesando en Popayán Solar AI: {text}"

demo = gr.Interface(fn=demo_predict, inputs="text", outputs="text", title="Solar Popayán Demo Port")
demo.launch(server_name="127.0.0.1", server_port=7860, prevent_thread_lock=True)
""")

print(f"⚙ Iniciando {APP_FILE} en segundo plano en el puerto {PORT}...")
# Lanzamos la app usando subprocess sin bloquear la celda actual
process = subprocess.Popen([sys.executable, APP_FILE], stdout=subprocess.PIPE, stderr=subprocess.PIPE)

# 4. Chequeo inteligente de puerto activo (evitando sleeps ciegos)
def is_port_open(port):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        return s.connect_ex(('127.0.0.1', port)) == 0

print("⏳ Esperando a que el servicio Gradio se inicialice y abra el puerto...")
max_retries = 30
port_opened = False
for i in range(max_retries):
    if is_port_open(PORT):
        port_opened = True
        break
    time.sleep(1)

if not port_opened:
    print(f"❌ ERROR: La aplicación no levantó en el puerto {PORT} después de {max_retries} segundos.")
    # Mostrar posibles logs de error de la app
    stdout, stderr = process.communicate(timeout=1)
    print("Logs del subproceso:", stderr.decode("utf-8"))
else:
    print(f"✔ Puerto {PORT} detectado activo.")

    # 5. Abrir el túnel con pyngrok
    try:
        # Cerrar túneles previos activos para evitar límites de cuenta gratuita
        tunnels = ngrok.get_tunnels()
        for t in tunnels:
            print(f"⚙ Cerrando túnel antiguo: {t.public_url}")
            ngrok.disconnect(t.public_url)

        print("⚙ Abriendo nuevo túnel público con ngrok...")
        public_url = ngrok.connect(PORT, "http")

        # 6. Mostrar URL pública final
        print("\n" + "="*80)
        print("🎉 ¡APLICACIÓN DESPLEGADA CON ÉXITO!")
        print(f"🔗 URL PÚBLICA (ngrok): {public_url}")
        print("="*80 + "\n")
    except Exception as e:
        print("❌ Error al conectar con ngrok:", str(e))

    # Mantener el proceso vivo para evitar que el Garbage Collector limpie el subproceso
    try:
        print("🛑 Presiona 'Detener Celda' (el botón cuadrado de play) en Colab para terminar el servicio.")
        while True:
            time.sleep(5)
    except KeyboardInterrupt:
        print("\n🔌 Apagando túneles y cerrando servidor en segundo plano...")
        ngrok.kill()
        process.kill()
        print("✔ Servicio finalizado exitosamente.")

⚙ Instalando pyngrok...
✔ pyngrok instalado correctamente.
✔ Token de ngrok configurado.
⚙ app.py no encontrado en el directorio. Creando un script demo de Gradio...
⚙ Iniciando app.py en segundo plano en el puerto 7860...
⏳ Esperando a que el servicio Gradio se inicialice y abra el puerto...
✔ Puerto 7860 detectado activo.
⚙ Abriendo nuevo túnel público con ngrok...

🎉 ¡APLICACIÓN DESPLEGADA CON ÉXITO!
🔗 URL PÚBLICA (ngrok): NgrokTunnel: "https://undoing-rebirth-scorch.ngrok-free.dev" -> "http://localhost:7860"

🛑 Presiona 'Detener Celda' (el botón cuadrado de play) en Colab para terminar el servicio.


```markdown
### 📝 Notas de Despliegue y Limitaciones de Entornos Gratuitos

* **Cambio de URL:** En el plan gratuito de ngrok, cada vez que reinicies el entorno o reconectes el túnel, se generará una URL aleatoria diferente.
* **Desconexión por Inactividad:** Google Colab desconectará automáticamente el backend si dejas de interactuar con la interfaz del notebook o tras un máximo de 12 horas de ejecución continua.

---

### 🚀 Extra: Migración futura a Streamlit Community Cloud (3 líneas)

Si en el futuro deseas reescribir tu interfaz a Streamlit y desplegarla permanentemente de forma gratuita:
1. Diseñas tu interfaz en un archivo `streamlit_app.py` utilizando contenedores y widgets de Streamlit (`st.title`, `st.selectbox`, etc.).
2. Subes el código de tu proyecto junto con un archivo `requirements.txt` que liste las dependencias a un repositorio público de GitHub.
3. Entras a [share.streamlit.io](https://share.streamlit.io/), inicias sesión con GitHub, seleccionas tu repositorio y tu aplicación quedará desplegada de forma continua las 24 horas del día.
```